In [ ]:
!pip install -U -q peft transformers accelerate safetensors

import os, gc, json, time, zipfile, math, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.set_float32_matmul_precision("high")

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 57.7 MB/s eta 0:00:00
DEVICE: cuda
GPU: Tesla T4
Mounted at /content/drive


In [ ]:
@dataclass
class CFGType:
    drive_root: str = "/content/drive/MyDrive"          # zip lives directly in MyDrive on this account
    zip_name: str = "ePillID_data.zip"
    extract_dir: str = "/content/extracted"
    data_root: str = ""

    dinov2_model_id: str = "facebook/dinov2-large"
    dinov2_model_key: str = "dinov2_large"

    num_workers: int = 0
    pin_memory: bool = False

    topk: Tuple[int, ...] = (1, 3, 5, 10, 20, 50)
    score_chunk_size: int = 256
    sidepair_aggs: Tuple[str, ...] = ("mean", "max")

    feature_batch_size: int = 48

    proj_embedding_dim: int = 512
    proj_hidden_dim: int = 1024

    seed: int = 42

CFG = CFGType()
DRIVE_ROOT = Path(CFG.drive_root)
ZIP_PATH = DRIVE_ROOT / CFG.zip_name
EXTRACT_DIR = Path(CFG.extract_dir)

# --- Run folders copied over from your old account's Drive ---
RUN_DIR_MAIN = DRIVE_ROOT / "ePillID_dinov2_adaptation_results" / "epillid_dinov2_fusion_finetune_lora_20260709_233943"
RUN_DIR_AUGHEAD = DRIVE_ROOT / "ePillID_dinov2_adaptation_results" / "epillid_dinov2_fusion_finetune_lora_20260708_234614"

AUG_HEAD_CKPT = RUN_DIR_AUGHEAD / "augmented_projection_head_200ep" / "best_projection_head.pt"
LORA_HEAD_CKPT = RUN_DIR_MAIN / "lora_fine_tuned_backbone" / "best_lora_head.pt"

# Feature cache paths — 224px (original) and 448px TTA (new, used for the accuracy boosters)
REF_FEAT_224_PATH = RUN_DIR_MAIN / "feature_cache" / f"{CFG.dinov2_model_key}_ref_features.pt"
REF_FEAT_448_PATH = RUN_DIR_MAIN / "persistent_aug_feature_cache" / "ref_feat_448_tta.pt"          # NEW, will be generated
TEST_FEAT_448_PATH = RUN_DIR_MAIN / "persistent_aug_feature_cache" / "test_feat_448_tta.pt"          # NEW, will be generated
VAL_FEAT_448_PATH = RUN_DIR_MAIN / "persistent_aug_feature_cache" / "val_feat_448_tta.pt"            # NEW, will be generated

for label, p in [
    ("ZIP_PATH", ZIP_PATH),
    ("AUG_HEAD_CKPT", AUG_HEAD_CKPT),
    ("LORA_HEAD_CKPT", LORA_HEAD_CKPT),
    ("REF_FEAT_224_PATH", REF_FEAT_224_PATH),
]:
    print(f"{label}: {p}  --  {'FOUND' if p.exists() else 'MISSING'}")


ZIP_PATH: /content/drive/MyDrive/ePillID_data.zip  --  FOUND
AUG_HEAD_CKPT: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260708_234614/augmented_projection_head_200ep/best_projection_head.pt  --  FOUND
LORA_HEAD_CKPT: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/lora_fine_tuned_backbone/best_lora_head.pt  --  FOUND
REF_FEAT_224_PATH: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/feature_cache/dinov2_large_ref_features.pt  --  FOUND


In [ ]:
def seed_everything(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def is_dir_empty(p: Path) -> bool:
    return not any(p.iterdir())

if is_dir_empty(EXTRACT_DIR):
    assert ZIP_PATH.exists(), f"Zip not found at {ZIP_PATH}"
    print(f"Extracting {ZIP_PATH} -> {EXTRACT_DIR} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Done.")
else:
    print("EXTRACT_DIR not empty, skipping extraction.")

def find_epillid_data_root(extract_dir: Path) -> Path:
    if CFG.data_root:
        p = Path(CFG.data_root)
        assert (p / "classification_data").exists(), p
        assert (p / "folds").exists(), p
        return p
    candidates = []
    for classif_dir in extract_dir.rglob("classification_data"):
        root = classif_dir.parent
        if (root / "folds").exists():
            candidates.append(root)
    if not candidates and (extract_dir / "classification_data").exists() and (extract_dir / "folds").exists():
        candidates.append(extract_dir)
    if not candidates:
        raise FileNotFoundError(f"Could not find classification_data/ and folds/ under {extract_dir}")
    return sorted(candidates, key=lambda p: len(str(p)))[0]

DATA_ROOT = find_epillid_data_root(EXTRACT_DIR)
CFG.data_root = str(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)

def find_default_split_files(data_root: Path):
    folds_dir = data_root / "folds"
    all_csvs = sorted(folds_dir.rglob("*_all.csv"))
    if not all_csvs:
        raise FileNotFoundError(f"No *_all.csv files found under {folds_dir}")
    preferred = [p for p in all_csvs if "pilltypeid_nih_sidelbls0.01_metric_5folds" in str(p)]
    all_csv = preferred[0] if preferred else all_csvs[0]
    base_dir = all_csv.parent
    stem = all_csv.name[:-len("_all.csv")]
    val_csv = base_dir / f"{stem}_3.csv"
    test_csv = base_dir / f"{stem}_4.csv"
    if not val_csv.exists() or not test_csv.exists():
        fold_csvs = sorted([p for p in base_dir.glob(f"{stem}_*.csv") if not p.name.endswith("_all.csv")])
        val_csv, test_csv = fold_csvs[-2], fold_csvs[-1]
    return all_csv, val_csv, test_csv

ALL_CSV, VAL_CSV, TEST_CSV = find_default_split_files(DATA_ROOT)
print("ALL_CSV:", ALL_CSV)
print("VAL_CSV:", VAL_CSV)
print("TEST_CSV:", TEST_CSV)

def to_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(int).astype(bool)
    return s.astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False, "yes": True, "no": False,
    }).fillna(False).astype(bool)

def resolve_image_root(df: pd.DataFrame, data_root: Path, rel_col: str) -> Path:
    sample = df[rel_col].dropna().astype(str).head(200).tolist()
    roots = [data_root, data_root / "classification_data", data_root / "classification_data" / "fcn_mix_weight"]
    best_root, best_hits = None, -1
    for root in roots:
        hits = sum((root / rel).exists() for rel in sample)
        if hits > best_hits:
            best_root, best_hits = root, hits
    if best_root is None or best_hits == 0:
        image_files = {p.name: p for p in (data_root / "classification_data").rglob("*") if p.is_file()}
        hits = sum(Path(rel).name in image_files for rel in sample)
        if hits == 0:
            raise FileNotFoundError("Could not resolve image paths from CSV metadata.")
        return Path("__FILENAME_LOOKUP__")
    return best_root

def add_abs_paths(df: pd.DataFrame, data_root: Path) -> pd.DataFrame:
    df = df.copy()
    rel_col = "image_path" if "image_path" in df.columns else "images"
    root = resolve_image_root(df, data_root, rel_col)
    if str(root) == "__FILENAME_LOOKUP__":
        image_files = {p.name: p for p in (data_root / "classification_data").rglob("*") if p.is_file()}
        df["abs_path"] = df[rel_col].astype(str).map(lambda x: str(image_files.get(Path(x).name, "")))
    else:
        df["abs_path"] = df[rel_col].astype(str).map(lambda x: str(root / x))
    df["rel_key"] = df[rel_col].astype(str)
    return df

def prepare_df(csv_path: Path, data_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = add_abs_paths(df, data_root)
    df["is_ref"] = to_bool_series(df["is_ref"])
    df["is_front"] = to_bool_series(df["is_front"]) if "is_front" in df.columns else True
    return df

all_df = prepare_df(ALL_CSV, DATA_ROOT)
val_df = prepare_df(VAL_CSV, DATA_ROOT)
test_df = prepare_df(TEST_CSV, DATA_ROOT)

label_candidates = ["label", "pilltype_id", "label_prod_code", "product_code"]
LABEL_COL = next((c for c in label_candidates if c in all_df.columns), None)

val_keys = set(val_df["rel_key"])
test_keys = set(test_df["rel_key"])
train_df = all_df[~all_df["rel_key"].isin(val_keys | test_keys)].copy()

ref_df = train_df[train_df["is_ref"]].copy()
val_query_df = val_df[~val_df["is_ref"]].copy()
test_query_df = test_df[~test_df["is_ref"]].copy()
if len(val_query_df) == 0: val_query_df = val_df.copy()
if len(test_query_df) == 0: test_query_df = test_df.copy()

label_encoder = LabelEncoder()
label_encoder.fit(all_df[LABEL_COL].astype(str))

def add_label_indexes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["label_str"] = df[LABEL_COL].astype(str)
    df["label_idx"] = label_encoder.transform(df["label_str"])
    return df

ref_df = add_label_indexes(ref_df)
val_query_df = add_label_indexes(val_query_df)
test_query_df = add_label_indexes(test_query_df)

N_CLASSES = len(label_encoder.classes_)
print("N_CLASSES:", N_CLASSES)
print("ref_df:", ref_df.shape, "val_query_df:", val_query_df.shape, "test_query_df:", test_query_df.shape)

Extracting /content/drive/MyDrive/ePillID_data.zip -> /content/extracted ...
Done.
DATA_ROOT: /content/extracted/ePillID_data
ALL_CSV: /content/extracted/ePillID_data/folds/pilltypeid_nih_sidelbls0.01_metric_5folds/base/pilltypeid_nih_sidelbls0.01_metric_5folds_all.csv
VAL_CSV: /content/extracted/ePillID_data/folds/pilltypeid_nih_sidelbls0.01_metric_5folds/base/pilltypeid_nih_sidelbls0.01_metric_5folds_3.csv
TEST_CSV: /content/extracted/ePillID_data/folds/pilltypeid_nih_sidelbls0.01_metric_5folds/base/pilltypeid_nih_sidelbls0.01_metric_5folds_4.csv
N_CLASSES: 4902
ref_df: (9804, 13) val_query_df: (745, 13) test_query_df: (745, 13)


In [ ]:
def label_score_matrix(query_emb, ref_emb, ref_labels, num_classes, chunk_size=256, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ref_emb = F.normalize(ref_emb.to(device), dim=1)
    ref_labels = ref_labels.to(device).long()
    all_scores = []
    for start in tqdm(range(0, query_emb.shape[0], chunk_size), desc="Score queries", leave=False):
        q = F.normalize(query_emb[start:start + chunk_size].to(device), dim=1)
        sim = q @ ref_emb.T
        b = sim.shape[0]
        scores = torch.full((b, num_classes), -1e9, device=device)
        index = ref_labels.unsqueeze(0).expand(b, -1)
        if hasattr(scores, "scatter_reduce_"):
            scores.scatter_reduce_(1, index, sim, reduce="amax", include_self=True)
        else:
            for c in torch.unique(ref_labels):
                scores[:, int(c.item())] = sim[:, ref_labels == c].max(dim=1).values
        all_scores.append(scores.cpu())
    return torch.cat(all_scores, dim=0)

def topk_accuracy_from_scores(scores: torch.Tensor, labels: torch.Tensor, topk=(1, 5, 10, 20, 50)):
    max_k = min(max(topk), scores.shape[1])
    _, pred = scores.topk(max_k, dim=1)
    labels_2d = labels.view(-1, 1)
    metrics = {}
    for k in topk:
        k_eff = min(k, scores.shape[1])
        metrics[f"top{k}_acc"] = (pred[:, :k_eff] == labels_2d).any(dim=1).float().mean().item()
    return metrics

def average_precision_from_scores(scores: torch.Tensor, labels: torch.Tensor):
    y_true = np.zeros(scores.shape, dtype=np.int8)
    y_true[np.arange(scores.shape[0]), labels.numpy()] = 1
    return float(average_precision_score(y_true.ravel(), scores.numpy().ravel()))

def rank_of_correct_vector(scores: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    correct_scores = scores[torch.arange(scores.shape[0]), labels]
    return (scores > correct_scores[:, None]).sum(dim=1) + 1

def build_query_info_from_df(df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "label_idx": torch.as_tensor(df["label_idx"].to_numpy(dtype=np.int64, copy=True), dtype=torch.long),
        "abs_path": df["abs_path"].astype(str).tolist(),
        "is_front": torch.as_tensor(df["is_front"].to_numpy(dtype=bool, copy=True), dtype=torch.bool),
        "label_str": df["label_str"].astype(str).tolist(),
    }

def build_two_side_pairs(query_info: Dict[str, Any], scores: torch.Tensor, agg: str = "mean"):
    qdf = pd.DataFrame({
        "row": np.arange(len(query_info["abs_path"])),
        "label_idx": query_info["label_idx"].numpy(),
        "label_str": query_info["label_str"],
        "abs_path": query_info["abs_path"],
        "is_front": query_info["is_front"].numpy().astype(bool),
    })
    front = qdf[qdf["is_front"]].copy()
    back = qdf[~qdf["is_front"]].copy()
    if len(front) == 0 or len(back) == 0:
        return None, None, None
    pairs = front.merge(back, on="label_idx", suffixes=("_front", "_back"))
    if len(pairs) == 0:
        return None, None, None
    front_rows = torch.as_tensor(pairs["row_front"].to_numpy(dtype=np.int64, copy=True), dtype=torch.long)
    back_rows = torch.as_tensor(pairs["row_back"].to_numpy(dtype=np.int64, copy=True), dtype=torch.long)
    front_scores = scores[front_rows]
    back_scores = scores[back_rows]
    pair_scores = torch.maximum(front_scores, back_scores) if agg == "max" else (front_scores + back_scores) / 2.0
    pair_labels = torch.as_tensor(pairs["label_idx"].to_numpy(dtype=np.int64, copy=True), dtype=torch.long)
    return pair_scores, pair_labels, pairs

def evaluate_scores(scores: torch.Tensor, query_info: Dict[str, Any], prefix: str, model_key: str, split: str = "test"):
    labels = query_info["label_idx"].long()
    metrics = {"model_key": model_key, "split": split}
    single = topk_accuracy_from_scores(scores, labels, CFG.topk)
    single["micro_ap"] = average_precision_from_scores(scores, labels)
    metrics.update({f"{prefix}_single_{k}": v for k, v in single.items()})
    for agg in CFG.sidepair_aggs:
        pair_scores, pair_labels, pairs = build_two_side_pairs(query_info, scores, agg=agg)
        if pair_scores is not None:
            pm = topk_accuracy_from_scores(pair_scores, pair_labels, CFG.topk)
            pm["micro_ap"] = average_precision_from_scores(pair_scores, pair_labels)
            pm["pair_count"] = int(pair_scores.shape[0])
            metrics.update({f"{prefix}_twoside_{agg}_{k}": v for k, v in pm.items()})
    return metrics

def make_predictions_df(scores: torch.Tensor, query_info: Dict[str, Any], model_key: str, split: str, top_k: int = 50):
    k = min(top_k, scores.shape[1])
    values, indices = scores.topk(k, dim=1)
    ranks = rank_of_correct_vector(scores, query_info["label_idx"])
    rows = []
    for i in range(scores.shape[0]):
        rows.append({
            "model_key": model_key, "split": split,
            "query_path": query_info["abs_path"][i],
            "query_label_idx": int(query_info["label_idx"][i]),
            "query_label": query_info["label_str"][i],
            "is_front": bool(query_info["is_front"][i]),
            "rank_of_correct": int(ranks[i]),
            "top_label_indices": json.dumps([int(x) for x in indices[i].tolist()]),
            "top_scores": json.dumps([float(x) for x in values[i].tolist()]),
        })
    return pd.DataFrame(rows)

test_query_info = build_query_info_from_df(test_query_df)
val_query_info = build_query_info_from_df(val_query_df)

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, num_classes: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_dim, out_dim),
        )
        self.classifier = nn.Linear(out_dim, num_classes)
        self.arc_weight = nn.Parameter(torch.empty(num_classes, out_dim))
        nn.init.xavier_uniform_(self.arc_weight)
    def forward(self, x, labels=None):
        emb = F.normalize(self.proj(x), dim=1)
        logits = self.classifier(emb)
        return {"emb": emb, "logits": logits, "arc_logits": None}

@torch.no_grad()
def apply_projection_head(head: nn.Module, info: Dict[str, Any], batch_size: int = 1024):
    head.eval()
    xs = info["emb"].float()
    outs = []
    for start in range(0, xs.shape[0], batch_size):
        x = xs[start:start + batch_size].to(DEVICE)
        outs.append(head(x)["emb"].cpu())
    return {**info, "emb": torch.cat(outs, dim=0)}

In [ ]:
class PillPILDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True).copy()
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "image": Image.open(row["abs_path"]).convert("RGB"),
            "label_idx": int(row["label_idx"]),
            "is_front": bool(row["is_front"]),
            "abs_path": row["abs_path"],
            "label_str": row["label_str"],
        }

def collate_pil(batch):
    return {
        "images": [b["image"] for b in batch],
        "label_idx": torch.tensor([b["label_idx"] for b in batch], dtype=torch.long),
        "is_front": torch.tensor([b["is_front"] for b in batch], dtype=torch.bool),
        "abs_path": [b["abs_path"] for b in batch],
        "label_str": [b["label_str"] for b in batch],
    }

def make_pil_loader(df, batch_size, shuffle=False):
    return DataLoader(PillPILDataset(df), batch_size=batch_size, shuffle=shuffle,
                       num_workers=CFG.num_workers, collate_fn=collate_pil)

import hashlib

def _df_fingerprint(df: pd.DataFrame) -> str:
    # Identifies WHICH images a cache was built from, not just how many --
    # a real run loaded a stale otc_ref_feat_448.pt/otc_query_feat_448.pt
    # left over from an earlier, smaller harvest (212 cached query rows vs.
    # 233 in the current otc_query_df) because the cache functions below
    # only checked cache_path.exists(), silently evaluating against the
    # wrong images. Recompute whenever the input set doesn't match what's
    # on disk instead of trusting any file that happens to already exist.
    return hashlib.sha1("\n".join(sorted(df["abs_path"].astype(str))).encode()).hexdigest()

@torch.no_grad()
def extract_dinov2_features_224(df: pd.DataFrame, cache_path: Path):
    fingerprint = _df_fingerprint(df)
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        if cached.get("_fingerprint") == fingerprint:
            print("Loading cached features:", cache_path)
            return cached
        print(f"Cache at {cache_path} doesn't match this {len(df)}-row input (stale from an earlier run) -- recomputing instead of trusting it.")
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(CFG.dinov2_model_id)
    model = AutoModel.from_pretrained(CFG.dinov2_model_id)
    model.to(DEVICE).eval()
    loader = make_pil_loader(df, batch_size=CFG.feature_batch_size, shuffle=False)
    embs, labels, paths, fronts, label_strs = [], [], [], [], []
    for batch in tqdm(loader, desc="Extract 224px DINOv2 features"):
        inputs = processor(images=batch["images"], return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            outputs = model(pixel_values=pixel_values)
            feat = outputs.pooler_output if getattr(outputs, "pooler_output", None) is not None else outputs.last_hidden_state[:, 0]
        embs.append(feat.float().cpu())
        labels.append(batch["label_idx"].cpu())
        fronts.append(batch["is_front"].cpu())
        paths.extend(batch["abs_path"])
        label_strs.extend(batch["label_str"])
    info = {"emb": torch.cat(embs, dim=0), "label_idx": torch.cat(labels, dim=0).long(),
            "is_front": torch.cat(fronts, dim=0).bool(), "abs_path": paths, "label_str": label_strs,
            "_fingerprint": fingerprint}
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(info, cache_path)
    del model, processor
    cleanup_cuda()
    return info

def _tta_views(img: Image.Image, crop_size: int = 448):
    """Deterministic TTA views: full resize, horizontal flip, and a mild zoomed center crop."""
    base = img.resize((crop_size, crop_size), Image.LANCZOS)
    flipped = base.transpose(Image.FLIP_LEFT_RIGHT)
    w, h = img.size
    zoom = 0.85
    cw, ch = int(w * zoom), int(h * zoom)
    left, top = (w - cw) // 2, (h - ch) // 2
    zoomed = img.crop((left, top, left + cw, top + ch)).resize((crop_size, crop_size), Image.LANCZOS)
    return [base, flipped, zoomed]

@torch.no_grad()
def extract_dinov2_features_448_tta(df: pd.DataFrame, cache_path: Path, batch_size: int = 16, n_views: int = 3):
    fingerprint = _df_fingerprint(df)
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        if cached.get("_fingerprint") == fingerprint:
            print("Loading cached features:", cache_path)
            return cached
        print(f"Cache at {cache_path} doesn't match this {len(df)}-row input "
              f"(stale from an earlier run) -- recomputing instead of trusting it.")
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(CFG.dinov2_model_id)
    model = AutoModel.from_pretrained(CFG.dinov2_model_id)
    model.to(DEVICE).eval()
    all_embs, all_labels, all_fronts, all_paths, all_label_strs = [], [], [], [], []
    for start in tqdm(range(0, len(df), batch_size), desc=f"Extract 448px TTA ({n_views} views)"):
        chunk = df.iloc[start:start + batch_size]
        chunk_embs = []
        for _, row in chunk.iterrows():
            img = Image.open(row["abs_path"]).convert("RGB")
            views = _tta_views(img, crop_size=448)[:n_views]
            inputs = processor(images=views, return_tensors="pt")
            pv = inputs["pixel_values"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                outputs = model(pixel_values=pv)
                emb = outputs.pooler_output if getattr(outputs, "pooler_output", None) is not None else outputs.last_hidden_state[:, 0]
            chunk_embs.append(emb.float().mean(dim=0, keepdim=True).cpu())
        all_embs.append(torch.cat(chunk_embs, dim=0))
        all_labels.append(torch.as_tensor(chunk["label_idx"].to_numpy(), dtype=torch.long))
        all_fronts.append(torch.as_tensor(chunk["is_front"].to_numpy(dtype=bool)))
        all_paths.extend(chunk["abs_path"].tolist())
        all_label_strs.extend(chunk["label_str"].tolist())
    info = {"emb": torch.cat(all_embs, dim=0), "label_idx": torch.cat(all_labels, dim=0),
            "is_front": torch.cat(all_fronts, dim=0), "abs_path": all_paths, "label_str": all_label_strs,
            "_fingerprint": fingerprint}
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(info, cache_path)
    del model, processor
    cleanup_cuda()
    return info

In [ ]:
ref_feat_224 = extract_dinov2_features_224(ref_df, REF_FEAT_224_PATH)
ref_feat_448 = extract_dinov2_features_448_tta(ref_df, REF_FEAT_448_PATH, n_views=3)     # NEW: resolution-matched reference set
test_feat_448 = extract_dinov2_features_448_tta(test_query_df, TEST_FEAT_448_PATH, n_views=3)
val_feat_448 = extract_dinov2_features_448_tta(val_query_df, VAL_FEAT_448_PATH, n_views=3)  # NEW: for tuning fusion weights

print("ref_feat_224:", ref_feat_224["emb"].shape)
print("ref_feat_448:", ref_feat_448["emb"].shape)
print("test_feat_448:", test_feat_448["emb"].shape)
print("val_feat_448:", val_feat_448["emb"].shape)

Loading cached features: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/feature_cache/dinov2_large_ref_features.pt
Loading cached features: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/persistent_aug_feature_cache/ref_feat_448_tta.pt
Loading cached features: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/persistent_aug_feature_cache/test_feat_448_tta.pt
Loading cached features: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/persistent_aug_feature_cache/val_feat_448_tta.pt
ref_feat_224: torch.Size([9804, 1024])
ref_feat_448: torch.Size([9804, 1024])
test_feat_448: torch.Size([745, 1024])
val_feat_448: torch.Size([745, 1024])


In [ ]:
in_dim = int(ref_feat_224["emb"].shape[1])

# Augmented projection head (200 epochs) — best single-model result: 85.37% single top5
head_aug = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES).to(DEVICE)
aug_ckpt = torch.load(AUG_HEAD_CKPT, map_location=DEVICE)
head_aug.load_state_dict(aug_ckpt["head_state_dict"])
head_aug.eval()
print("Loaded augmented head from:", AUG_HEAD_CKPT)

# LoRA-trained head
head_lora = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES).to(DEVICE)
lora_ckpt = torch.load(LORA_HEAD_CKPT, map_location=DEVICE)
head_lora.load_state_dict(lora_ckpt["head_state_dict"])
head_lora.eval()
print("Loaded LoRA head from:", LORA_HEAD_CKPT)

Loaded augmented head from: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260708_234614/augmented_projection_head_200ep/best_projection_head.pt
Loaded LoRA head from: /content/drive/MyDrive/ePillID_dinov2_adaptation_results/epillid_dinov2_fusion_finetune_lora_20260709_233943/lora_fine_tuned_backbone/best_lora_head.pt


In [ ]:
with torch.no_grad():
    ref_aug_proj_448 = apply_projection_head(head_aug, ref_feat_448)
    test_aug_proj = apply_projection_head(head_aug, test_feat_448)


# **V3: MULTI-DATABASE EXPANSION — OTC + prescription pills, many manufacturers**

**Goal.** The model above is a *retrieval* system, not a closed-set classifier: at
inference time we never use `head.classifier` / `head.arc_weight` (see
`label_score_matrix` above and `backend/app/classifier.py` in the deployed app)
— we only compare the projected embedding of a query image against a gallery
of reference embeddings per class, and take the best-matching class. **That
means we can add brand-new pill types to the gallery without retraining
anything**, as long as we can get reference photos + a label for them. This
section does exactly that using DailyMed (`dailymed.nlm.nih.gov`), the NIH/FDA's
live, continuously-updated repository of Structured Product Labeling (SPL) —
unlike the now-discontinued NIH Pillbox/C3PI corpus that ePillID's original
4,902 classes already come from, DailyMed covers both OTC and prescription
products from hundreds of labelers/manufacturers and includes actual product
photographs ("media" files), not just box art.

**Honest scope note.** "Guaranteed top-10 no matter what it is" isn't
achievable for pills that have zero reference photos anywhere — there's
nothing to match against. What this section targets is: **90%+ top-5 across
the union of ePillID's 4,902 classes and whatever DailyMed products this
harvest collects photos for** — realistically a couple thousand additional
classes (see the bulk-download section below for the actual yield math), not
literally "tens of thousands."

**Pipeline in this section:**
1. Download and parse DailyMed's bulk SPL archives (OTC + prescription),
   filtering to solid-oral products with usable photos.
2. Build an OTC/Rx reference + held-out query split, extract the same DINOv2
   448px TTA features and project them with `head_aug` (the head actually
   shipped in the FastAPI backend — see `deploy_model.py`/`classifier.py`).
3. Merge the ePillID and new-drug reference galleries and evaluate top-k
   accuracy on each subset (regression check + new-capability check).
4. Export deployment artifacts in the **exact** format
   `backend/app/classifier.py` expects.


In [ ]:
# ============================================================================
# V3.0: Config shared by the harvest pipeline (used by the bulk-download path below)
# ============================================================================
from dataclasses import dataclass

@dataclass
class OTCCFGType:
    api_base: str = "https://dailymed.nlm.nih.gov/dailymed/services/v2"
    allowed_dosage_forms: tuple = (
        "TABLET", "CAPSULE", "CAPLET", "CHEWABLE", "LOZENGE", "GEL CAP", "GELCAP",
    )
    min_pill_photo_score: float = 0.55      # CLIP zero-shot "is this a pill photo" threshold
    clip_model_id: str = "openai/clip-vit-base-patch32"
    otc_query_holdout_per_class: int = 1    # held-out images/class for eval; rest -> reference gallery

    otc_cache_root: str = ""
    otc_images_dir: str = ""

OTC_CFG = OTCCFGType()
OTC_CFG.otc_cache_root = str(RUN_DIR_MAIN / "otc_dailymed")
OTC_CFG.otc_images_dir = str(Path(OTC_CFG.otc_cache_root) / "images")
Path(OTC_CFG.otc_images_dir).mkdir(parents=True, exist_ok=True)
print("OTC cache root:", OTC_CFG.otc_cache_root)


### Bulk-download path (the harvest method used in this notebook)

DailyMed publishes pre-packaged bulk ZIP archives (`human_otc`, `human_rx`)
containing every SPL's XML and submitted images. Downloading a handful of
large files and parsing them locally avoids DailyMed's per-item REST API
rate limits entirely, and it's the only practical way to reach prescription
drugs at scale.

**Archive structure:** each outer zip extracts into a single `otc/` or
`prescription/` folder containing thousands of individual per-SPL zips
(`{date}_{setid}.zip`). V3.B2 smoke-tests this before the full parse — if
its output doesn't look like real SPL data, stop and check
`BULK_CFG.manual_zip_urls` before running V3.B3.

**Key design points:**
- Downloads/extraction happen on local Colab disk
  (`/content/otc_dailymed_bulk`), not Drive — the full corpus is tens of GB,
  well past Drive's free 15GB quota. Only the final exported artifacts
  (V3.9/V3.10) go to Drive.
- V3.B3 opens each per-SPL zip in memory and does CLIP (pill-vs-packaging) +
  blur filtering inline, deleting each zip immediately after processing, so
  the ~143K-file corpus never needs to exist on disk all at once.
- Singleton-image classes (most SPLs submit exactly one photo) are kept in
  the reference gallery — retrieval only needs one reference embedding per
  class. Classes with ≥2 images are additionally held out for the accuracy
  check in V3.8/V3.9.
- Brand/generic/labeler names are **not** extracted from the SPL XML directly
  (the `<name>` element ordering isn't reliable enough to trust). Only the
  NDC is extracted here; V3.10 resolves the real drug name for each NDC via
  NIH's RxNav API instead.
- Already harvested before? Use the checkpoint-restore cell below instead of
  re-running V3.B0-V3.B3.

**Realistic expectation:** across the full ~143,533-zip corpus, roughly
1.6-2% of SPLs yield a qualifying pill-photo class, which caps this harvest
at around 2,300-2,900 total new classes — not tens of thousands.


### Already harvested before? Restore the checkpoint instead of re-running V3.B0-V3.B3

If `harvest_checkpoint.json` already exists on your Drive (saved by the
checkpoint cell near the end of the bulk-download section), **run the cell
below and then skip straight to V3.6** -- do not re-run V3.B0 through V3.B3,
that would re-download and re-parse everything from scratch for no reason.
The harvested images are already sitting on Drive (`KEPT_IMAGES_DIR`, under
`OTC_CFG.otc_images_dir`), so nothing needs to be re-fetched.

If you don't have a checkpoint yet, skip this cell and run V3.B0 onward
normally -- it'll create one for you near the end.


In [ ]:
# Restore a previous harvest -- requires OTC_CFG (run V3.0 first, no network
# calls there) but NOT the rest of the bulk pipeline.
CHECKPOINT_PATH = Path(OTC_CFG.otc_cache_root) / "harvest_checkpoint.json"
if CHECKPOINT_PATH.exists():
    _restored = json.load(open(CHECKPOINT_PATH))
    qualifying_products = _restored["qualifying_products"]
    media_by_setid = _restored["media_by_setid"]
    kept = set(_restored["kept"])
    # attempted_setids tracks every SPL already looked at (qualifying or not),
    # so a later V3.B3 run skips it instead of re-parsing the same early zips
    # every session. Older checkpoints (saved before this field existed) fall
    # back to just the setids that qualified -- rejected ones from that run
    # will get looked at again once, which is harmless.
    attempted_setids = set(_restored.get("attempted_setids", media_by_setid.keys()))
    by_domain = {}
    for p in qualifying_products:
        by_domain[p["domain"]] = by_domain.get(p["domain"], 0) + 1
    print(f"Restored from checkpoint: {len(qualifying_products)} qualifying products ({by_domain}), "
          f"{len(media_by_setid)} SPLs, {len(kept)} kept images, {len(attempted_setids)} SPLs already "
          f"looked at (qualifying or not).")
    print("Want more classes? Re-run V3.B0-V3.B3 to keep harvesting (it resumes past what's already "
          "attempted, doesn't redo it) -- then re-run the checkpoint-save cell before V3.6. Otherwise "
          "skip straight to V3.6 with what you have.")
else:
    print(f"No checkpoint found at {CHECKPOINT_PATH} -- run V3.B0 onward to harvest from scratch.")


In [ ]:
# ============================================================================
# V3.B0: Config for the bulk-download path (now: Rx + OTC release groups)
# ============================================================================
# Self-contained import (time/json/zipfile/Path already come from cell 0).
import urllib.request, urllib.error, urllib.parse
from dataclasses import dataclass, field

@dataclass
class BulkReleaseGroup:
    name: str                # "rx" | "otc" -- also used to label harvested classes for V3.8's report
    zip_link_pattern: str     # regex matched against the index page's HTML

@dataclass
class BulkCFGType:
    index_page_url: str = "https://dailymed.nlm.nih.gov/dailymed/spl-resources-all-drug-labels.cfm"
    # Rx listed first: `download_with_progress` below spends the shared time
    # budget in this order, and Rx is what actually covers the drugs doctors
    # flagged (levothyroxine, rosuvastatin, rabeprazole) -- OTC only fills
    # whatever budget is left over once Rx is done.
    release_groups: tuple = (
        BulkReleaseGroup("rx", r'href="([^"]*human[_-]?rx[^"]*\.zip)"'),
        BulkReleaseGroup("otc", r'href="([^"]*human[_-]?otc[^"]*\.zip)"'),
    )
    # Fallback if auto-discovery finds nothing: {"rx": ["https://...zip", ...], "otc": [...]}
    manual_zip_urls: dict = field(default_factory=dict)

    download_time_budget_seconds: int = 5400    # 90 min ceiling, shared across both groups (Rx first)
    parse_time_budget_seconds: int = 1800        # 30 min ceiling per run -- V3.B3/V3.9 are resumable
                                                  # (each processed zip is deleted immediately), so
                                                  # reaching the full ~143,533-zip corpus means rerunning
                                                  # V3.B3 across multiple sessions, not raising this to an
                                                  # unsafe single-run length.
    max_spls_to_process: int = 150000            # was 60000 -- that capped BELOW the full corpus (143,533
                                                  # zips), artificially limiting the ceiling on new classes.
                                                  # Raised above the corpus size so it's never the limiting
                                                  # factor; the real bottleneck is repeated runs over time,
                                                  # not this number. Observed yield so far (~1.6-2% of
                                                  # zips processed become a qualifying class) implies
                                                  # roughly 2,300-2,900 classes if the ENTIRE corpus gets
                                                  # processed -- treat "3,000-5,000" as an optimistic
                                                  # upper range contingent on the unprocessed portion
                                                  # yielding at least as well as what's been seen, not a
                                                  # guarantee.

    allowed_dosage_forms: tuple = (
        "TABLET", "CAPSULE", "CAPLET", "CHEWABLE", "LOZENGE", "GEL CAP", "GELCAP",
    )
    # Edge-variance floor for the post-CLIP blur filter (V3.5) -- a sharp
    # macro pill photo scores in the hundreds/thousands on this metric, a
    # genuinely out-of-focus one scores well under 30 in spot checks. Not a
    # calibrated clinical threshold, just a coarse "unusable" cutoff -- spot
    # check dropped images the first time you run this.
    blur_variance_threshold: float = 30.0
    request_timeout: int = 30

    # Reclaim disk space once a zip's contents are safely extracted -- Rx+OTC
    # bulk archives total tens of GB, so keeping every zip AND its extracted
    # contents on disk simultaneously roughly doubles peak usage for no
    # reason once extraction succeeds.
    delete_zip_after_extract: bool = True
    # Stop downloading/extracting gracefully (same pattern as the time
    # budgets below) once free disk drops under this, instead of crashing
    # with a bare "no space left on device" mid-write.
    min_free_disk_gb: float = 5.0

    bulk_root: str = ""

BULK_CFG = BulkCFGType()
# Deliberately LOCAL Colab disk (/content/...), NOT Google Drive.
# RUN_DIR_MAIN lives under /content/drive/MyDrive/... -- a real run hit a
# Google Drive storage-quota error after downloading just 3 of 6 Rx parts
# (~9.7GB): Drive's free tier is 15GB shared across your whole Google
# account, and Rx+OTC bulk archives total tens of GB, so writing this much
# there was never going to fit. Local Colab disk is a separate, usually
# much larger quota, and it's fine for this pipeline to be ephemeral (lost
# on disconnect) -- only the final, much smaller exported artifacts
# (V3.9/V3.10, a few hundred MB at most) need to persist to Drive; the raw
# zips and extracted SPL folders don't.
BULK_CFG.bulk_root = "/content/otc_dailymed_bulk"
Path(BULK_CFG.bulk_root).mkdir(parents=True, exist_ok=True)
print("Bulk root (local Colab disk, NOT Google Drive):", BULK_CFG.bulk_root)
print(f"Release groups (priority order): {[g.name for g in BULK_CFG.release_groups]}")
print(f"Disk floor: {BULK_CFG.min_free_disk_gb} GB -- downloads/extraction stop gracefully below this, "
      f"same as the time budgets do.")


In [ ]:
# ============================================================================
# V3.B1: Discover and download the DailyMed bulk archive(s) for each release
#        group (Rx, then OTC). Resumable (skips files already downloaded)
#        and time-budgeted -- stops after download_time_budget_seconds
#        total, keeping whatever parts finished. Rx is downloaded first
#        because it's the release group that actually contains the drugs
#        doctors flagged; if the budget runs out partway through, you keep
#        all completed Rx parts and whatever OTC parts fit in what's left.
# ============================================================================
import re, shutil

def discover_bulk_zip_urls_for_group(group: "BulkReleaseGroup") -> list:
    manual = BULK_CFG.manual_zip_urls.get(group.name)
    if manual:
        return list(manual)
    try:
        req = urllib.request.Request(BULK_CFG.index_page_url, headers={"User-Agent": "pill-id-research/1.0"})
        with urllib.request.urlopen(req, timeout=BULK_CFG.request_timeout) as resp:
            html = resp.read().decode("utf-8", errors="ignore")
    except Exception as e:
        print(f"Could not fetch the download index page: {e}")
        return []
    matches = re.findall(group.zip_link_pattern, html, flags=re.IGNORECASE)
    # The index page links each file TWICE -- an absolute https mirror and an
    # ftp:// mirror -- and the old "doesn't start with http -> prepend the
    # domain" logic mangled the ftp entries into garbage like
    # "https://dailymed.nlm.nih.govftp://...". Only keep real absolute
    # http(s) URLs (or genuine site-relative paths starting with "/"), and
    # dedupe by filename so the same zip part doesn't appear twice.
    urls = []
    seen_files = set()
    for m in matches:
        if m.startswith("http://") or m.startswith("https://"):
            url = m
        elif m.startswith("/"):
            url = f"https://dailymed.nlm.nih.gov{m}"
        else:
            continue  # e.g. an ftp:// mirror link -- the https mirror covers the same file
        fname = url.split("/")[-1].split("?")[0]
        if fname in seen_files:
            continue
        seen_files.add(fname)
        urls.append(url)
    return sorted(urls)

bulk_zip_urls_by_group = {}
for group in BULK_CFG.release_groups:
    urls = discover_bulk_zip_urls_for_group(group)
    bulk_zip_urls_by_group[group.name] = urls
    print(f"Discovered {len(urls)} '{group.name}' bulk zip URL(s):")
    for u in urls:
        print(" ", u)

if not any(bulk_zip_urls_by_group.values()):
    print(f"\n⚠ Auto-discovery found nothing. Go to {BULK_CFG.index_page_url} , find the "
          "'Human Prescription Drug Labels' and 'Human OTC Drug Labels' zip link(s) yourself, then set e.g.:\n"
          "   BULK_CFG.manual_zip_urls = {'rx': ['https://...part1.zip', ...], 'otc': ['https://...part1.zip', ...]}\n"
          "and rerun this cell.")

BULK_ZIP_DIR = Path(BULK_CFG.bulk_root) / "zips"
BULK_ZIP_DIR.mkdir(parents=True, exist_ok=True)

def _free_disk_gb(path) -> float:
    return shutil.disk_usage(path).free / 1e9

def download_with_progress(url: str, dest: Path, deadline: float) -> bool:
    if dest.exists():
        print(f"  Already downloaded: {dest.name} ({dest.stat().st_size / 1e6:.0f} MB)")
        return True
    if time.time() > deadline:
        return False
    free_gb = _free_disk_gb(dest.parent)
    if free_gb < BULK_CFG.min_free_disk_gb:
        print(f"  ⏹ Only {free_gb:.1f} GB free (floor: {BULK_CFG.min_free_disk_gb} GB) -- skipping {url.split('/')[-1]}.")
        return False
    tmp = dest.with_suffix(".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "pill-id-research/1.0"})
        with urllib.request.urlopen(req, timeout=BULK_CFG.request_timeout) as resp:
            total = int(resp.headers.get("Content-Length", 0))
            downloaded = 0
            chunk_count = 0
            with open(tmp, "wb") as f:
                while True:
                    if time.time() > deadline:
                        print(f"  ⏱ Time budget hit mid-download of {url.split('/')[-1]} -- partial file discarded.")
                        tmp.unlink(missing_ok=True)
                        return False
                    chunk_count += 1
                    if chunk_count % 200 == 0 and _free_disk_gb(dest.parent) < BULK_CFG.min_free_disk_gb:
                        print(f"  ⏹ Disk floor hit mid-download of {url.split('/')[-1]} -- partial file discarded.")
                        tmp.unlink(missing_ok=True)
                        return False
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
            tmp.rename(dest)
        print(f"  Downloaded {dest.name}: {downloaded / 1e6:.0f} MB"
              + (f" of {total / 1e6:.0f} MB expected" if total else ""))
        return True
    except Exception as e:
        print(f"  Failed to download {url}: {e}")
        tmp.unlink(missing_ok=True)
        return False

deadline = time.time() + BULK_CFG.download_time_budget_seconds
downloaded_zips = []  # list of (group_name, Path)
total_discovered = sum(len(v) for v in bulk_zip_urls_by_group.values())
stopped_for_disk = False

for group in BULK_CFG.release_groups:
    for url in bulk_zip_urls_by_group.get(group.name, []):
        fname = url.split("/")[-1].split("?")[0]
        dest = BULK_ZIP_DIR / fname
        ok = download_with_progress(url, dest, deadline)
        if ok:
            downloaded_zips.append((group.name, dest))
        elif _free_disk_gb(BULK_ZIP_DIR) < BULK_CFG.min_free_disk_gb:
            stopped_for_disk = True
        if time.time() > deadline or stopped_for_disk:
            break
    if time.time() > deadline or stopped_for_disk:
        n_rx = sum(1 for g, _ in downloaded_zips if g == "rx")
        n_otc = sum(1 for g, _ in downloaded_zips if g == "otc")
        reason = "disk space floor" if stopped_for_disk else "download time budget"
        print(f"⏹ Stopped ({reason}) -- proceeding with {len(downloaded_zips)} part(s) downloaded so far ({n_rx} rx, {n_otc} otc).")
        break

n_rx = sum(1 for g, _ in downloaded_zips if g == "rx")
n_otc = sum(1 for g, _ in downloaded_zips if g == "otc")
print(f"\n{len(downloaded_zips)}/{total_discovered} archive part(s) downloaded ({n_rx} rx, {n_otc} otc).")
if n_rx == 0:
    print("⚠ No Rx parts downloaded -- the merged gallery will be OTC-only again, same as before this "
          "change. If Rx discovery found 0 URLs above, the index page's Rx link text/pattern likely "
          "differs from what BULK_CFG.release_groups assumes -- inspect the page and set "
          "BULK_CFG.manual_zip_urls['rx'] yourself.")
if stopped_for_disk:
    print(f"⚠ Stopped due to low disk space (floor: {BULK_CFG.min_free_disk_gb} GB). V3.B2 next "
          "reclaims space as each zip is extracted (BULK_CFG.delete_zip_after_extract defaults to "
          "True) -- rerun this cell afterward to fetch more parts with the freed space.")


In [ ]:
# ============================================================================
# V3.B2: Extract the downloaded archive(s) and SANITY-CHECK the internal
#        layout before the expensive full parse in V3.B3. Read this cell's
#        output carefully -- see the caveat above. Also records which
#        release group (rx/otc) each SPL folder came from, since folder
#        names alone (bare SETIDs) don't carry that information --
#        V3.B3/V3.6/V3.8 use this to report Rx and OTC accuracy separately.
# ============================================================================
import zipfile

BULK_EXTRACT_DIR = Path(BULK_CFG.bulk_root) / "extracted"
BULK_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

setid_domain = dict(setid_domain) if "setid_domain" in dir() else {}

for group_name, zip_path in downloaded_zips:
    marker = BULK_EXTRACT_DIR / f".extracted_{zip_path.stem}"
    already_extracted = marker.exists()
    if already_extracted:
        print(f"Already extracted: {zip_path.name} ({group_name})")
        if not zip_path.exists():
            # Deleted by a previous pass (delete_zip_after_extract) -- domain
            # info for this zip's setids was already recorded then.
            continue
    else:
        print(f"Extracting {zip_path.name} ({group_name}) ...")

    # Read namelist() for domain-tracking BEFORE any deletion below -- the
    # zip has to still exist on disk for this to work.
    with zipfile.ZipFile(zip_path, "r") as zf:
        if not already_extracted:
            zf.extractall(BULK_EXTRACT_DIR)
            marker.touch()
        top_level_names = {n.split("/")[0] for n in zf.namelist() if n.strip("/")}
    for name in top_level_names:
        setid_domain[name] = group_name

    # Reclaim disk space now that this zip's contents are safely extracted --
    # Rx+OTC archives total tens of GB, keeping zip + extracted copy of
    # everything at once roughly doubles peak disk usage for no benefit.
    if not already_extracted and BULK_CFG.delete_zip_after_extract and zip_path.exists():
        size_gb = zip_path.stat().st_size / 1e9
        zip_path.unlink()
        print(f"  Deleted {zip_path.name} after extraction (freed {size_gb:.1f} GB).")

all_entries = sorted(BULK_EXTRACT_DIR.iterdir())
print(f"\n{len(all_entries)} top-level entries under {BULK_EXTRACT_DIR}")
print("First 5:", [p.name for p in all_entries[:5]])

# Expected convention: one directory per SPL (named by SETID), each
# containing that SPL's XML plus any images it shipped with.
sample_dirs = [p for p in all_entries if p.is_dir()][:3]
if not sample_dirs:
    print("\n⚠ No subdirectories found at the top level -- the archive may have a flatter or "
          "deeper structure than expected (e.g. XML files directly at top level, or an extra "
          "nesting level). Print `all_entries` yourself and adjust V3.B3's walk logic before "
          "proceeding -- do not run V3.B3 as-is if this warning appears.")
else:
    for d in sample_dirs:
        contents = list(d.iterdir())
        xml_files = [p for p in contents if p.suffix.lower() == ".xml"]
        img_files = [p for p in contents if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".gif")]
        dom = setid_domain.get(d.name, "unknown")
        print(f"\n{d.name}/  [{dom}]  ({len(contents)} files: {len(xml_files)} xml, {len(img_files)} image)")
        print("  ", [p.name for p in contents[:8]])
    print("\nIf the above shows real XML + image files per folder, V3.B3 is safe to run as-is. "
          "If it looks wrong (no .xml files, unexpected nesting), stop here and report back what "
          "you see instead of running the full parse.")


In [ ]:
# ============================================================================
# V3.B3 (rewritten): DailyMed's bulk release nests one level deeper than
# documented -- each top-level "otc"/"prescription" folder contains tens of
# thousands of INDIVIDUAL PER-SPL ZIP FILES ({date}_{setid}.zip), not raw
# XML+images directly. A real run's V3.B2 smoke test caught this before the
# old parser wasted the parse budget getting zero results. This version
# opens each per-SPL zip in memory (zipfile reads a member's bytes without
# extracting to disk), parses its XML there, and -- only for products that
# pass the dosage-form/NDC filter -- scores its images with CLIP + a blur
# check entirely in memory too. Only images that pass BOTH filters ever get
# written to disk, into KEPT_IMAGES_DIR. This folds in what was previously
# a separate V3.5 pass (which now detects this and skips itself) specifically
# to avoid ever extracting the full ~143K-zip corpus to disk before
# filtering it -- that would recreate the disk-exhaustion risk V3.B1/V3.B2
# were just fixed for, at a much larger scale (a real run needed a 90 min
# window just to download+extract the 17 outer archives; extracting all
# ~143K inner zips too, unfiltered, would be an order of magnitude worse).
# Trade-off, stated plainly: CLIP runs per-SPL (1-4 images per call) rather
# than in large cross-SPL batches, which is less GPU-throughput-efficient
# than V3.5's original batched design -- simpler and disk-safe was chosen
# over maximally fast, given max_spls_to_process and the time budget still
# bound total work either way.
# ============================================================================
import xml.etree.ElementTree as ET
import io, re as _re
from collections import Counter
from transformers import CLIPModel, CLIPProcessor
from PIL import ImageFilter

NDC_OID = "2.16.840.1.113883.6.69"  # standard HL7 OID for NDC codes

def _local(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def _setid_from_zip_name(name: str) -> str:
    # "20250109_2b1becb3-1a75-1497-e063-6394a90aea03.zip" -> the setid
    stem = Path(name).stem
    return _re.sub(r"^\d{8}_", "", stem)

def parse_spl_zip_metadata(zf: "zipfile.ZipFile"):
    xml_names = [n for n in zf.namelist() if n.lower().endswith(".xml")]
    if not xml_names:
        return None
    try:
        root = ET.fromstring(zf.read(xml_names[0]))
    except ET.ParseError:
        return None

    ndc = None
    for el in root.iter():
        if _local(el.tag) == "code" and el.get("codeSystem") == NDC_OID and el.get("code"):
            ndc = el.get("code")
            break
    if not ndc:
        return None

    dosage_form = None
    for el in root.iter():
        if _local(el.tag) == "formCode" and el.get("displayName"):
            dosage_form = el.get("displayName")
            break

    # NOTE: brand/generic/labeler names are intentionally NOT extracted here.
    # An earlier version grabbed every <name> element anywhere in the SPL
    # document (labeler org, active ingredient, every inactive
    # ingredient/excipient) and assigned them by raw document position --
    # a real harvest showed that assumption is wrong: 79% of entries ended
    # up with a manufacturer name (e.g. "American Health Packaging") in
    # brand_name/generic_name, and 23% had a random inactive excipient
    # (e.g. "Sucrose") in labeler_name instead of the actual drug. NDC
    # extraction above uses a completely different, reliable method (OID
    # match), so naming is deferred entirely to the NDC->RxNav lookup in
    # V3.10 -- see notebooks/fix_harvest_ndc_names.py for the same
    # after-the-fact fix applied to an already-harvested checkpoint.
    image_names = [n for n in zf.namelist() if Path(n).suffix.lower() in (".jpg", ".jpeg", ".png")]
    return {
        "ndc": ndc, "dosage_form": dosage_form, "brand_name": None,
        "generic_name": None, "labeler_name": None,
        "image_names": image_names,
    }

# CLIP + blur filter -- loaded once, applied inline per qualifying product
# (this is what used to be a separate V3.5 pass -- see that cell's skip-guard).
clip_model = CLIPModel.from_pretrained(OTC_CFG.clip_model_id).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(OTC_CFG.clip_model_id)
POSITIVE_PROMPTS = [
    "a close-up photo of a single pill or tablet on a plain background",
    "a macro photograph of a medication tablet or capsule showing its imprint",
]
NEGATIVE_PROMPTS = [
    "a photo of a drug package, box, or bottle",
    "a scanned image of a text label or package insert",
]
PROMPTS = POSITIVE_PROMPTS + NEGATIVE_PROMPTS
N_POS = len(POSITIVE_PROMPTS)

def _clip_pooled(output):
    """CLIPModel.get_text_features()/get_image_features() return a bare
    tensor in older transformers, but a BaseModelOutputWithPooling (with the
    actual projected embedding placed in .pooler_output) in newer ones -- a
    real run hit this exact break (AttributeError: 'BaseModelOutputWithPooling'
    object has no attribute 'norm') on transformers 5.14.1, confirmed against
    that version's modeling_clip.py source: get_text_features/get_image_features
    are both decorated @can_return_tuple and, by default, now return the full
    output object rather than the tensor the old code assumed. Handled
    defensively here since Colab's transformers version isn't pinned and can
    change independently of this notebook."""
    return output.pooler_output if hasattr(output, "pooler_output") else output

# Text prompts never change across SPLs -- embed them ONCE here, not inside
# the per-item loop below (the original per-call recompute was harmless but
# wasteful at up to 60,000 calls).
with torch.no_grad():
    _text_inputs_once = clip_processor(text=PROMPTS, return_tensors="pt", padding=True).to(DEVICE)
    _TEXT_EMB = F.normalize(_clip_pooled(clip_model.get_text_features(**_text_inputs_once)), dim=1)

@torch.no_grad()
def _clip_scores_for_images(pil_images):
    if not pil_images:
        return np.array([])
    img_inputs = clip_processor(images=pil_images, return_tensors="pt").to(DEVICE)
    img_emb = F.normalize(_clip_pooled(clip_model.get_image_features(**img_inputs)), dim=1)
    sim = (img_emb @ _TEXT_EMB.T).cpu().numpy()
    pos = sim[:, :N_POS].max(axis=1)
    neg = sim[:, N_POS:].max(axis=1)
    return 1.0 / (1.0 + np.exp(-(pos - neg) * 10.0))

_blur_threshold = BULK_CFG.blur_variance_threshold

def _blur_variance_from_pil(img) -> float:
    try:
        gray = img.convert("L").resize((256, 256))
        edges = np.asarray(gray.filter(ImageFilter.FIND_EDGES), dtype=np.float32)
        return float(edges.var())
    except Exception:
        return 0.0

KEPT_IMAGES_DIR = Path(OTC_CFG.otc_images_dir) / "kept_raw"
KEPT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Domain folder names as actually observed by V3.B2's smoke test on a real
# run -- "prescription" for Rx (not "rx"), "otc" for OTC. Checked
# defensively (both "prescription" and "rx") in case a future release uses
# a different name.
DOMAIN_FOLDERS = [
    ("rx", [d for d in ("prescription", "rx") if (BULK_EXTRACT_DIR / d).is_dir()]),
    ("otc", [d for d in ("otc",) if (BULK_EXTRACT_DIR / d).is_dir()]),
]

spl_zip_paths = []  # list of (domain, Path) -- Rx first, matching the
                     # existing "Rx matters more" prioritization
for domain, folder_names in DOMAIN_FOLDERS:
    for folder_name in folder_names:
        folder = BULK_EXTRACT_DIR / folder_name
        for p in folder.iterdir():
            if p.suffix.lower() == ".zip":
                spl_zip_paths.append((domain, p))

print(f"{len(spl_zip_paths)} per-SPL zip files found "
      f"({sum(1 for d,_ in spl_zip_paths if d=='rx')} rx, {sum(1 for d,_ in spl_zip_paths if d=='otc')} otc); "
      f"processing up to {BULK_CFG.max_spls_to_process}")

# Tighter disk floor than the download/extraction phases (BULK_CFG.min_free_disk_gb,
# 5.0 GB) -- deliberately: each iteration below deletes its own per-SPL zip
# right after reading it, so free space is reclaimed continuously in small
# increments (~hundreds of KB at a time across ~143K files) rather than
# needing the WHOLE remaining corpus's disk footprint free upfront. A real
# run hit exactly that problem: the 5.0 GB floor tripped before a single
# item was even processed, because nothing had been deleted yet to earn
# any of that space back. A smaller floor is safe here since a single
# item's write (a handful of filtered survivor images, typically KB-scale)
# can't plausibly consume a large chunk of it in one step.
PARSE_MIN_FREE_DISK_GB = 1.5

deadline = time.time() + BULK_CFG.parse_time_budget_seconds
qualifying_products = list(qualifying_products) if "qualifying_products" in dir() else []
media_by_setid = dict(media_by_setid) if "media_by_setid" in dir() else {}
kept = set(kept) if "kept" in dir() else set()
# Every setid ever looked at, qualifying or not -- lets a later run (this
# session or a fresh one, after restoring the checkpoint) skip straight to
# NEW zips instead of re-parsing the same early ones every time.
attempted_setids = set(attempted_setids) if "attempted_setids" in dir() else set()
n_parsed = n_skipped_already_attempted = n_kept_products = n_kept_images = 0
n_kept_by_domain = {"rx": 0, "otc": 0}
# Why a zip got rejected -- tells you whether low yield is "most SPLs
# genuinely have no usable pill photo" vs. an over-strict filter.
n_rejected_by_reason = Counter()
timed_out = stopped_for_disk = False

for domain, zip_path in tqdm(spl_zip_paths[: BULK_CFG.max_spls_to_process], desc="Parsing per-SPL zips"):
    if time.time() > deadline:
        timed_out = True
        break

    setid = _setid_from_zip_name(zip_path.name)
    if setid in attempted_setids:
        n_skipped_already_attempted += 1
        zip_path.unlink(missing_ok=True)
        continue
    n_parsed += 1
    attempted_setids.add(setid)
    kept_paths_this_spl = []
    reject_reason = None
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            parsed = parse_spl_zip_metadata(zf)
            if parsed is None:
                reject_reason = "no_ndc_or_unparseable_xml"
            elif not parsed["dosage_form"] or not any(
                f in parsed["dosage_form"].upper() for f in BULK_CFG.allowed_dosage_forms
            ):
                reject_reason = "wrong_dosage_form"
                parsed = None
            elif not parsed["image_names"]:
                reject_reason = "no_images_in_spl"
                parsed = None
            else:
                pil_images, valid_names = [], []
                for name in parsed["image_names"]:
                    try:
                        img = Image.open(io.BytesIO(zf.read(name))).convert("RGB")
                    except Exception:
                        continue
                    pil_images.append(img)
                    valid_names.append(name)
                if not pil_images:
                    reject_reason = "images_present_but_undecodable"
                else:
                    clip_scores = _clip_scores_for_images(pil_images)
                    for name, img, score in zip(valid_names, pil_images, clip_scores):
                        if score < OTC_CFG.min_pill_photo_score:
                            continue
                        if _blur_variance_from_pil(img) < _blur_threshold:
                            continue
                        dest = KEPT_IMAGES_DIR / f"{parsed['ndc']}_{setid}_{Path(name).name}"
                        img.save(dest)
                        kept_paths_this_spl.append(str(dest))
                    if not kept_paths_this_spl:
                        reject_reason = "failed_clip_or_blur_filter"
    except (zipfile.BadZipFile, OSError):
        parsed = None
        reject_reason = "bad_zip"
    finally:
        # Reclaim this zip's disk space now that we've read whatever we
        # needed from it -- unconditional, whether it qualified or not.
        # This is the fix: leaving ~143K small zips on disk until the very
        # end (the old behavior) is what exhausted disk before any parsing
        # could even start on a real run.
        zip_path.unlink(missing_ok=True)

    if shutil.disk_usage(KEPT_IMAGES_DIR).free / 1e9 < PARSE_MIN_FREE_DISK_GB:
        stopped_for_disk = True
        break

    if parsed is None or not kept_paths_this_spl:
        n_rejected_by_reason[reject_reason or "unknown"] += 1
        continue

    qualifying_products.append({
        "setid": setid, "ndc": parsed["ndc"], "brand_name": parsed["brand_name"],
        "generic_name": parsed["generic_name"], "labeler_name": parsed["labeler_name"],
        "domain": domain,
    })
    media_by_setid[setid] = kept_paths_this_spl
    kept.update(kept_paths_this_spl)
    n_kept_products += 1
    n_kept_images += len(kept_paths_this_spl)
    n_kept_by_domain[domain] += len(kept_paths_this_spl)

del clip_model, clip_processor
cleanup_cuda()

if n_skipped_already_attempted:
    print(f"Skipped {n_skipped_already_attempted} zips already attempted in a previous run.")

if timed_out:
    print(f"⏱ Stopped after the {BULK_CFG.parse_time_budget_seconds/60:.0f}-minute parse budget -- "
          f"parsed {n_parsed} NEW per-SPL zips this pass (kept {n_kept_products} qualifying "
          f"products / {n_kept_images} images). Rerun this cell later (after restoring/saving the "
          f"checkpoint) to keep going -- it'll pick up where this pass left off, not from the start.")
elif stopped_for_disk:
    print(f"⏹ Stopped -- free disk dropped under {PARSE_MIN_FREE_DISK_GB} GB (even after deleting each "
          f"processed zip) after parsing {n_parsed}/{len(spl_zip_paths)} per-SPL zips, kept "
          f"{n_kept_products} qualifying products / {n_kept_images} images this pass. If this happens "
          f"repeatedly with very few kept, something upstream (not this loop) is eating disk -- check "
          f"`!du -sh /content/otc_dailymed_bulk/*` for what's actually using space.")
else:
    print(f"Parsed {n_parsed} per-SPL zips (all available, or hit the {BULK_CFG.max_spls_to_process} "
          f"cap), kept {n_kept_products} qualifying products / {n_kept_images} images.")

print(f"qualifying_products: {len(qualifying_products)} total | media_by_setid: {len(media_by_setid)} SPLs "
      f"| kept images: {len(kept)}")
print(f"  kept images by domain this pass: {n_kept_by_domain}")
if n_rejected_by_reason:
    print("  rejected this pass, by reason:")
    for reason, count in n_rejected_by_reason.most_common():
        print(f"    {reason}: {count} ({count / n_parsed:.1%} of parsed)")
if n_kept_products == 0 and n_parsed > 0:
    print("\n⚠ Parsed zips but kept zero products -- either the dosage-form/NDC filter or the CLIP/blur "
          "filter is rejecting everything. Open one real per-SPL zip directly (e.g. "
          "zipfile.ZipFile(spl_zip_paths[0][1]).namelist()) and compare against the extraction logic "
          "above before continuing.")
else:
    print("\nImages are already CLIP + blur filtered (done inline above) -- V3.5 will detect this and "
          "skip its own pass. Continue with V3.6.")


In [ ]:
# ============================================================================
# V3.B3.5 (checkpoint): save the harvest's bookkeeping to Google Drive so it
# survives a Colab disconnect/runtime restart. The kept (filtered) images
# themselves already live on Drive -- KEPT_IMAGES_DIR is under
# OTC_CFG.otc_images_dir, which is Drive-backed (unlike BULK_CFG.bulk_root,
# deliberately moved to local disk earlier to avoid the storage-quota crash
# the RAW/unfiltered corpus caused). What's NOT safe is the in-memory index
# (qualifying_products/media_by_setid/kept) connecting each image back to
# its NDC/setid -- that only exists in this session's RAM right now. JSON,
# not pickle, so it's readable/portable regardless of Python version.
#
# Safe to run mid-pipeline: it only reads the existing variables and writes
# a file, nothing here changes what V3.B3/V3.6 onward see -- rerun V3.B3
# again (same or a later session) and then re-run this cell to update the
# checkpoint with more coverage (e.g. once it reaches OTC, still 0 so far).
# ============================================================================
import json as _json

CHECKPOINT_PATH = Path(OTC_CFG.otc_cache_root) / "harvest_checkpoint.json"
with open(CHECKPOINT_PATH, "w") as f:
    _json.dump({
        "qualifying_products": qualifying_products,
        "media_by_setid": media_by_setid,
        "kept": sorted(kept),
        "attempted_setids": sorted(attempted_setids),
    }, f)

by_domain = {}
for p in qualifying_products:
    by_domain[p["domain"]] = by_domain.get(p["domain"], 0) + 1

print(f"Checkpoint saved: {CHECKPOINT_PATH}")
print(f"  {len(qualifying_products)} qualifying products ({by_domain}) | "
      f"{len(media_by_setid)} SPLs | {len(kept)} kept images")
print(f"  Images themselves are already durable on Drive under {KEPT_IMAGES_DIR} -- "
      f"this file is just the index connecting them back to NDC/setid.")
print("\nTo restore in a FRESH session later (after re-running cells 0-13, V3.0, and V3.B0-V3.B2):")
print("  _restored = json.load(open(CHECKPOINT_PATH))")
print("  qualifying_products = _restored['qualifying_products']")
print("  media_by_setid = _restored['media_by_setid']")
print("  kept = set(_restored['kept'])")
print("  attempted_setids = set(_restored.get('attempted_setids', media_by_setid.keys()))")
print("  # then continue with V3.B3 (it will extend these, not overwrite) or skip straight to V3.6.")


In [ ]:
# ============================================================================
# V3.5: Pill-vs-packaging + blur filtering already happened inline in V3.B3
# (or came pre-filtered from a restored checkpoint) -- this just confirms
# `kept` is populated before continuing to V3.6.
# ============================================================================
assert "kept" in dir() and kept, "Run V3.B0-V3.B3 (or restore a checkpoint) before this cell."
print(f"{len(kept)} images already CLIP + blur filtered. Continue with V3.6.")


### V3.5.5: Narrow the harvest to the most common OTC + Rx drugs

The full-corpus harvest above found real classes, but most are long-tail
products a doctor or patient will rarely encounter. This cell resolves
every qualifying product's NDC to its real generic name (via RxNav) and
keeps only the ones matching a curated list of high-volume drugs: ClinCalc's
Top-300 most-prescribed Rx generics (fetched live, with a hardcoded
fallback) plus a curated list of common OTC actives. Everything from V3.6
onward then automatically operates on this smaller, clinically-relevant
subset instead of the full long tail.


In [ ]:
# ============================================================================
# V3.5.5: Filter the harvest down to the MOST COMMON OTC + Rx drugs.
#         Resolves each qualifying product's NDC to its real generic name
#         via RxNav (the same method V3.10 uses later -- done here first so
#         V3.10 can just reuse ndc_lookup instead of re-fetching), then
#         keeps only classes whose name matches a curated common-drug list.
#         qualifying_products_all keeps the unfiltered set around in case
#         you want to widen COMMON_RX_DRUGS/COMMON_OTC_DRUGS and rerun.
# ============================================================================
import re as _re
import urllib.error
from concurrent.futures import ThreadPoolExecutor as _TPE, as_completed as _as_completed

RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"


class _RateLimited(Exception):
    pass


def _rxnav_get(url, timeout=20, retries=5):
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "pill-id-research/1.0"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.load(resp)
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise _RateLimited(f"{url}: {last_err}")


def _rxnav_candidates(token):
    parts = token.split("-")
    cands = []
    if len(parts) >= 2:
        cands.append(f"{parts[0].zfill(5)}-{parts[1].zfill(4)}")
    cands.append(token)
    seen = set()
    return [c for c in cands if not (c in seen or seen.add(c))]


def _rxnav_name(rxcui, cache):
    if rxcui in cache:
        return cache[rxcui]
    data = _rxnav_get(f"{RXNAV_BASE}/rxcui/{rxcui}/property.json?propName=RxNorm%20Name")
    pc = data.get("propConceptGroup", {}).get("propConcept", [])
    name = pc[0]["propValue"] if pc else None
    cache[rxcui] = name
    return name


def _rxnav_find_prop(props, *substrings):
    for prop_name, value in props.items():
        if any(s in prop_name.upper() for s in substrings) and value:
            return value
    return None


def resolve_ndc_via_rxnav(token, rx_cache):
    for cand in _rxnav_candidates(token):
        data = _rxnav_get(f"{RXNAV_BASE}/ndcproperties.json?id={cand}&ndcstatus=ALL")
        pl = data.get("ndcPropertyList", {}).get("ndcProperty", [])
        if not pl:
            continue
        p = pl[0]
        props = {x["propName"]: x["propValue"] for x in p.get("propertyConceptList", {}).get("propertyConcept", [])}
        rxcui = p.get("rxcui") or None
        name = _rxnav_name(rxcui, rx_cache) if rxcui else None
        if not name:
            continue
        return {
            "name": name, "rxcui": rxcui,
            "imprint": props.get("IMPRINT_CODE") or None,
            "color": props.get("COLORTEXT") or None,
            "status": props.get("NDC_STATUS") or None,
            "shape": _rxnav_find_prop(props, "SHAPE"),
            "score": _rxnav_find_prop(props, "SCORE"),
        }
    return None


# ---- 1. Build the common-drug name list -----------------------------------
# Hardcoded fallback (~120 well-known high-volume US generics) used only if
# ClinCalc's live Top-300 page isn't reachable -- keeps this cell working
# even if that specific site is down or blocked from wherever this runs.
_FALLBACK_TOP_RX = [
    "atorvastatin", "levothyroxine", "metformin", "lisinopril", "amlodipine", "metoprolol",
    "albuterol", "omeprazole", "losartan", "gabapentin", "hydrochlorothiazide", "sertraline",
    "simvastatin", "montelukast", "escitalopram", "rosuvastatin", "bupropion", "trazodone",
    "furosemide", "pantoprazole", "fluticasone", "tamsulosin", "amoxicillin", "azithromycin",
    "prednisone", "tramadol", "duloxetine", "citalopram", "clopidogrel", "warfarin",
    "alprazolam", "clonazepam", "cyclobenzaprine", "meloxicam", "venlafaxine", "allopurinol",
    "spironolactone", "glipizide", "carvedilol", "hydroxyzine", "ropinirole", "quetiapine",
    "aripiprazole", "lamotrigine", "topiramate", "propranolol", "atenolol", "diltiazem",
    "verapamil", "clonidine", "doxycycline", "cephalexin", "ciprofloxacin", "levofloxacin",
    "sulfamethoxazole", "nitrofurantoin", "fluoxetine", "paroxetine", "mirtazapine",
    "buspirone", "zolpidem", "temazepam", "lorazepam", "diazepam", "methylphenidate",
    "amphetamine", "atomoxetine", "rabeprazole", "esomeprazole", "lansoprazole", "ranitidine",
    "famotidine", "sildenafil", "tadalafil", "finasteride", "tamoxifen", "letrozole",
    "anastrozole", "methotrexate", "hydroxychloroquine", "prednisolone", "budesonide",
    "amoxicillin clavulanate", "clindamycin", "metronidazole", "valacyclovir", "acyclovir",
    "oseltamivir", "glimepiride", "pioglitazone", "sitagliptin", "empagliflozin",
    "dapagliflozin", "semaglutide", "apixaban", "rivaroxaban", "digoxin", "amiodarone",
    "potassium chloride", "ferrous sulfate", "folic acid", "pregabalin", "baclofen",
    "tizanidine", "methocarbamol", "naproxen", "ibuprofen", "morphine", "oxycodone",
    "hydrocodone", "buspirone hydrochloride", "olanzapine", "risperidone", "ondansetron",
    "promethazine", "metoclopramide", "losartan potassium", "valsartan", "irbesartan",
    "olmesartan", "telmisartan", "candesartan", "benazepril", "enalapril", "ramipril",
    "fosinopril", "quinapril", "trandolapril", "nebivolol", "bisoprolol", "labetalol",
    "sotalol", "isosorbide mononitrate", "hydralazine", "minoxidil", "doxazosin", "terazosin",
    "nifedipine", "felodipine", "ezetimibe", "fenofibrate", "gemfibrozil", "niacin",
    "colesevelam", "chlorthalidone", "indapamide", "torsemide", "bumetanide", "triamterene",
    "amiloride", "eplerenone", "dronedarone", "flecainide", "propafenone", "dofetilide",
    "ticagrelor", "prasugrel", "dabigatran", "edoxaban", "phenytoin", "levetiracetam",
    "valproic acid", "carbamazepine", "oxcarbazepine", "zonisamide", "lacosamide", "primidone",
    "perampanel", "donepezil", "memantine", "rivastigmine", "galantamine",
    "levodopa carbidopa", "amantadine", "entacapone", "rasagiline", "selegiline",
    "benztropine", "trihexyphenidyl", "dantrolene", "carisoprodol", "orphenadrine",
    "metaxalone", "chlorzoxazone", "sumatriptan", "rizatriptan", "zolmitriptan", "eletriptan",
    "naratriptan", "almotriptan", "divalproex sodium", "milnacipran", "desvenlafaxine",
    "nefazodone", "vilazodone", "vortioxetine", "fluvoxamine", "amitriptyline",
    "nortriptyline", "imipramine", "desipramine", "doxepin", "clomipramine",
    "lithium carbonate", "quetiapine fumarate", "ziprasidone", "paliperidone", "asenapine",
    "lurasidone", "cariprazine", "brexpiprazole", "clozapine", "haloperidol", "fluphenazine",
    "perphenazine", "chlorpromazine", "prochlorperazine", "granisetron", "dolasetron",
    "palonosetron", "aprepitant", "scopolamine", "dimenhydrinate", "zaleplon", "eszopiclone",
    "ramelteon", "suvorexant", "lemborexant", "triazolam", "flurazepam", "chlordiazepoxide",
    "oxazepam", "clorazepate", "dexmethylphenidate", "lisdexamfetamine", "guanfacine",
    "modafinil", "armodafinil", "glyburide", "rosiglitazone", "saxagliptin", "linagliptin",
    "alogliptin", "canagliflozin", "ertugliflozin", "acarbose", "repaglinide", "nateglinide",
    "liothyronine", "methimazole", "propylthiouracil", "dexamethasone",
    "fluticasone propionate", "mometasone", "beclomethasone", "ciclesonide", "triamcinolone",
    "levalbuterol", "ipratropium", "tiotropium", "umeclidinium", "formoterol", "salmeterol",
    "indacaterol", "olodaterol", "zafirlukast", "zileuton", "theophylline", "roflumilast",
    "esomeprazole magnesium", "dexlansoprazole", "cimetidine", "nizatidine", "sucralfate",
    "misoprostol", "dicyclomine", "hyoscyamine", "mesalamine", "sulfasalazine", "balsalazide",
    "azathioprine", "diphenoxylate atropine", "lactulose", "magnesium citrate", "ampicillin",
    "penicillin v potassium", "dicloxacillin", "cefuroxime", "cefdinir", "cefpodoxime",
    "ceftriaxone", "cefazolin", "clarithromycin", "erythromycin", "minocycline",
    "tetracycline", "moxifloxacin", "ofloxacin", "trimethoprim sulfamethoxazole", "vancomycin",
    "linezolid", "daptomycin", "fluconazole", "itraconazole", "terbinafine", "ketoconazole",
    "nystatin", "famciclovir", "rimantadine", "isoniazid", "rifampin", "ethambutol",
    "pyrazinamide", "cilostazol", "pentoxifylline", "dutasteride", "vardenafil", "oxybutynin",
    "tolterodine", "solifenacin", "mirabegron", "darifenacin", "trospium", "testosterone",
    "estradiol", "conjugated estrogens", "medroxyprogesterone", "progesterone",
    "norethindrone", "levonorgestrel", "exemestane", "leflunomide", "febuxostat", "colchicine",
    "probenecid", "alendronate sodium", "risedronate", "ibandronate", "zoledronic acid",
    "calcitriol", "cholecalciferol", "ergocalciferol", "cyanocobalamin", "tretinoin",
    "adapalene", "benzoyl peroxide", "clindamycin phosphate", "mupirocin", "permethrin",
    "ivermectin", "selenium sulfide", "betamethasone dipropionate", "clobetasol propionate",
    "fluocinonide", "desonide", "calcipotriene", "tacrolimus ointment", "pimecrolimus",
    "latanoprost", "timolol ophthalmic", "brimonidine", "dorzolamide", "travoprost",
    "bimatoprost", "ofloxacin ophthalmic", "moxifloxacin ophthalmic",
    "prednisolone acetate ophthalmic", "cyclosporine ophthalmic", "ranolazine",
    "isosorbide dinitrate", "nitroglycerin", "amlodipine benazepril",
    "valsartan hydrochlorothiazide", "lisinopril hydrochlorothiazide",
    "losartan hydrochlorothiazide", "metoprolol hydrochlorothiazide", "diclofenac",
    "celecoxib", "indomethacin", "ketorolac", "etodolac", "nabumetone", "sulindac",
    "piroxicam", "oxycodone acetaminophen", "methadone", "buprenorphine",
    "buprenorphine naloxone", "naloxone", "medroxyprogesterone acetate", "ospemifene",
    "raloxifene", "cefixime", "cefadroxil", "doxycycline monohydrate", "naltrexone",
    "disulfiram", "acamprosate", "varenicline", "nicotine", "hydroxyzine pamoate",
    "diphenhydramine hydrochloride", "meclizine hydrochloride", "prochlorperazine maleate",
    "promethazine dm", "guaifenesin codeine", "benzonatate", "codeine",
    "hydrocodone chlorpheniramine", "phenazopyridine", "nitrofurantoin monohydrate",
    "sulfacetamide", "clindamycin benzoyl peroxide", "adapalene benzoyl peroxide",
    "erythromycin topical", "metronidazole topical", "azelaic acid", "hydroquinone",
    "minoxidil topical", "finasteride topical", "estradiol patch", "testosterone gel",
    "progesterone micronized", "levonorgestrel emergency", "ella", "clomiphene",
    "letrozole fertility", "spironolactone hydrochlorothiazide",
    "amiloride hydrochlorothiazide", "triamterene hydrochlorothiazide",
    "atenolol chlorthalidone", "bisoprolol hydrochlorothiazide"
]
_COMMON_OTC = [
    "acetaminophen", "ibuprofen", "aspirin", "naproxen sodium", "naproxen", "diphenhydramine",
    "loratadine", "cetirizine", "fexofenadine", "famotidine", "omeprazole", "esomeprazole",
    "loperamide", "docusate sodium", "bisacodyl", "simethicone", "calcium carbonate",
    "guaifenesin", "dextromethorphan", "pseudoephedrine", "phenylephrine", "melatonin",
    "meclizine", "bismuth subsalicylate", "senna", "magnesium hydroxide",
    "chlorpheniramine maleate", "doxylamine succinate", "polyethylene glycol", "ranitidine",
    "clotrimazole", "miconazole", "hydrocortisone", "benzocaine", "menthol", "camphor",
    "pyrantel pamoate", "clotrimazole betamethasone", "terbinafine hydrochloride",
    "tolnaftate", "miconazole nitrate", "hydrocortisone acetate", "pramoxine", "lidocaine",
    "zinc oxide", "aluminum hydroxide magnesium hydroxide",
    "calcium carbonate magnesium hydroxide", "sennosides", "psyllium", "methylcellulose",
    "castor oil", "mineral oil", "glycerin", "sodium phosphate", "aspirin caffeine",
    "acetaminophen aspirin caffeine", "ibuprofen pseudoephedrine",
    "acetaminophen pseudoephedrine", "triprolidine pseudoephedrine", "brompheniramine",
    "clemastine", "diphenhydramine phenylephrine", "guaifenesin dextromethorphan",
    "guaifenesin pseudoephedrine", "menthol camphor", "xylometazoline", "oxymetazoline",
    "sodium chloride nasal", "cromolyn sodium", "loperamide simethicone",
    "omeprazole magnesium", "calcium carbonate simethicone",
    "aluminum magnesium hydroxide simethicone", "attapulgite", "bisacodyl senna",
    "docusate sodium senna", "vitamin c", "vitamin d3", "vitamin b12", "biotin",
    "multivitamin", "iron polysaccharide", "glucosamine chondroitin", "fish oil",
    "melatonin extended release", "diphenhydramine acetaminophen", "vitamin b6", "vitamin e",
    "vitamin b complex", "magnesium oxide", "magnesium citrate supplement",
    "potassium citrate", "zinc sulfate", "calcium citrate", "coenzyme q10",
    "turmeric curcumin", "probiotic", "elderberry", "echinacea", "ginger",
    "activated charcoal", "loratadine d", "fexofenadine d", "cetirizine d",
    "acetaminophen diphenhydramine", "naproxen sodium pm", "ibuprofen pm", "aspirin low dose",
    "docusate calcium", "sodium bicarbonate antacid", "calcium magnesium zinc",
    "prenatal vitamin", "children's acetaminophen", "children's ibuprofen",
    "saline nasal spray", "artificial tears"
]


def _fetch_clincalc_top300():
    try:
        req = urllib.request.Request("https://clincalc.com/DrugStats/Top300Drugs.aspx",
                                      headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            html = resp.read().decode("utf-8", errors="ignore")
        cells_html = _re.findall(r"<td[^>]*>([^<]+)</td>", html)
        names = [c.strip().lower() for c in cells_html
                 if _re.fullmatch(r"[a-zA-Z][a-zA-Z \-/]{2,40}", c.strip())]
        names = [n for n in names if n not in ("rank", "drug name", "generic name")]
        if len(names) < 100:
            print(f"ClinCalc page fetched OK but only {len(names)} plausible drug-name cells parsed out "
                  f"(needed >=100) -- page structure may not match what this regex expects; using the "
                  f"hardcoded fallback instead.")
            return None
        return sorted(set(names))
    except Exception as e:
        print(f"Couldn't fetch ClinCalc Top 300 live ({e}) -- using the hardcoded fallback list instead.")
        return None


_live_top_rx = _fetch_clincalc_top300()
COMMON_RX_DRUGS = set(_live_top_rx) if _live_top_rx else set(_FALLBACK_TOP_RX)
COMMON_OTC_DRUGS = set(_COMMON_OTC)
COMMON_DRUG_NAMES = COMMON_RX_DRUGS | COMMON_OTC_DRUGS
print(f"Common-drug list: {len(COMMON_RX_DRUGS)} Rx generics "
      f"({'live from ClinCalc' if _live_top_rx else 'hardcoded fallback'}) + "
      f"{len(COMMON_OTC_DRUGS)} OTC actives = {len(COMMON_DRUG_NAMES)} total names to match against.")

# ---- 2. Resolve every qualifying product's NDC to a real name via RxNav ---
NDC_NAMES_IN = Path("/content/ndc_names.json")   # optional: upload the existing table here first
if "ndc_lookup" not in dir():
    ndc_lookup = json.load(open(NDC_NAMES_IN)) if NDC_NAMES_IN.exists() else {}
    print(f"Loaded {len(ndc_lookup)} existing NDC names" if ndc_lookup
          else "No existing ndc_names.json found -- starting fresh")

tokens_to_resolve = sorted({p["ndc"] for p in qualifying_products if p.get("ndc") and p["ndc"] not in ndc_lookup})
print(f"Resolving {len(tokens_to_resolve)} NDCs via RxNav "
      f"(skipping {len(qualifying_products) - len(tokens_to_resolve)} already resolved)...")

_rx_cache = {}
_resolved = _rate_limited = 0
with _TPE(max_workers=5) as pool:
    futures = {pool.submit(resolve_ndc_via_rxnav, t, _rx_cache): t for t in tokens_to_resolve}
    for i, fut in enumerate(_as_completed(futures), 1):
        token = futures[fut]
        try:
            info = fut.result()
            if info:
                ndc_lookup[token] = info
                _resolved += 1
        except _RateLimited:
            _rate_limited += 1
        if i % 200 == 0:
            print(f"  {i}/{len(tokens_to_resolve)} processed, {_resolved} resolved")
print(f"Resolved {_resolved}/{len(tokens_to_resolve)} new NDCs this run"
      + ("." if not _rate_limited else f" ({_rate_limited} rate-limited -- rerun this cell to retry)."))

# ---- 3. Filter qualifying_products/media_by_setid/kept to common drugs ---
def _is_common_drug(ndc):
    info = ndc_lookup.get(ndc)
    if not info or not info.get("name"):
        return False
    name_lower = info["name"].lower()
    return any(common in name_lower for common in COMMON_DRUG_NAMES)


_before = len(qualifying_products)
qualifying_products_all = qualifying_products  # unfiltered set, kept around for later widening
qualifying_products = [p for p in qualifying_products if _is_common_drug(p["ndc"])]
_kept_setids = {p["setid"] for p in qualifying_products}
media_by_setid = {sid: files for sid, files in media_by_setid.items() if sid in _kept_setids}
kept = {p for files in media_by_setid.values() for p in files}

by_domain = {}
for p in qualifying_products:
    by_domain[p["domain"]] = by_domain.get(p["domain"], 0) + 1
print(f"\nFiltered to common drugs: {_before} -> {len(qualifying_products)} qualifying products ({by_domain}), "
      f"{len(media_by_setid)} SPLs, {len(kept)} images.")
print("qualifying_products_all still holds the full unfiltered set -- to widen coverage later, add more "
      "names to COMMON_RX_DRUGS/COMMON_OTC_DRUGS above and rerun just this cell (no re-harvesting needed).")

# ---- Coverage check: which target common drugs actually ended up with a
# real class in the dataset, vs. having zero usable photo anywhere in the
# DailyMed harvest? This is the actual answer to "are the common drugs
# really in here," not just "did we filter for them."
_resolved_names_lower = [info["name"].lower() for info in ndc_lookup.values() if info and info.get("name")]
_matched_common_names = {name for name in COMMON_DRUG_NAMES
                          if any(name in resolved for resolved in _resolved_names_lower)}
_missing_common_names = sorted(COMMON_DRUG_NAMES - _matched_common_names)
print(f"\nCoverage check: {len(_matched_common_names)}/{len(COMMON_DRUG_NAMES)} target common drugs have "
      f"at least one class in the dataset (Rx: {len(_matched_common_names & COMMON_RX_DRUGS)}/{len(COMMON_RX_DRUGS)}, "
      f"OTC: {len(_matched_common_names & COMMON_OTC_DRUGS)}/{len(COMMON_OTC_DRUGS)}).")
if _missing_common_names:
    print(f"{len(_missing_common_names)} common drugs have NO matching class at all -- DailyMed's bulk "
          f"corpus likely has no usable photo for these specific ones (not a filter-list gap):")
    print(" ", _missing_common_names)

# Specifically check the drugs the original pilot feedback flagged as
# unrecognized -- these matter more than the aggregate coverage number.
_pilot_flagged_drugs = ["levothyroxine", "rosuvastatin", "rabeprazole"]
print("\nPilot-flagged drugs, checked against the FULL unfiltered harvest (not just the common-drug "
      "filtered set) to see whether the gap is upstream (never harvested) or just this filter step:")
for drug in _pilot_flagged_drugs:
    matches = [(ndc, info["name"]) for ndc, info in ndc_lookup.items()
               if info and info.get("name") and drug in info["name"].lower()]
    if matches:
        filtered_ndcs = {p["ndc"] for p in qualifying_products}
        in_filtered = any(ndc in filtered_ndcs for ndc, _ in matches)
        print(f"  {drug}: {len(matches)} match(es) in the full harvest, "
              f"{'INCLUDED in the filtered set' if in_filtered else 'present in the full harvest but NOT in the filtered set (matching bug worth checking)'} "
              f"-- e.g. {matches[0][1]!r}")
    else:
        print(f"  {drug}: 0 matches anywhere in the full harvest -- DailyMed's bulk corpus never yielded "
              f"a usable photo for this drug (rejected upstream by dosage-form/CLIP filtering in V3.B3, "
              f"or no SPL for it at all). This needs a different data source, not a filter-list fix.")


In [ ]:
# ============================================================================
# Batch version of the single-drug DailyMed check -- runs across EVERY common
# drug the coverage check flagged as missing, to quantify the real gap:
# genuinely has no photo on DailyMed at all, vs. has one the bulk harvest
# missed. Bounded and low-concurrency (5 workers, same as the proven-safe
# RxNav resolution pattern) -- NOT the broad, high-concurrency multi-drug
# crawl that got rate-limited earlier. Uses only /spls.json (confirmed
# working) and /spls/{setid}/media.json -- never the broken
# /spls/{setid}.json detail endpoint that returned HTTP 415.
# ============================================================================
import urllib.request, urllib.parse, json as _json, time
from concurrent.futures import ThreadPoolExecutor, as_completed

_API_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"


def _dm_get_batch(url, retries=4):
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "pill-id-research/1.0", "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                return _json.load(resp)
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{url}: {last_err}")


def check_drug_has_photos(drug_name: str, sample_size: int = 3) -> dict:
    params = urllib.parse.urlencode({"drug_name": drug_name, "pagesize": 100, "page": 1})
    try:
        discovery = _dm_get_batch(f"{_API_BASE}/spls.json?{params}")
    except Exception as e:
        return {"drug": drug_name, "status": "lookup_failed", "n_spls": None, "has_photos": None, "error": str(e)}
    candidates = discovery.get("data", [])
    if not candidates:
        return {"drug": drug_name, "status": "no_spls_at_all", "n_spls": 0, "has_photos": False}
    for c in candidates[:sample_size]:
        setid = c.get("setid")
        if not setid:
            continue
        try:
            media = _dm_get_batch(f"{_API_BASE}/spls/{setid}/media.json")
            media_files = media.get("data", {}).get("media", []) if isinstance(media.get("data"), dict) else media.get("data", [])
            if media_files:
                return {"drug": drug_name, "status": "has_photos", "n_spls": len(candidates), "has_photos": True, "setid": setid}
        except Exception:
            continue
    return {"drug": drug_name, "status": "spls_but_no_photos_sampled", "n_spls": len(candidates), "has_photos": False}


print(f"Checking {len(_missing_common_names)} missing common drugs directly against DailyMed's REST API "
      f"(bounded to a small sample of SPLs per drug, 5 concurrent workers)...")

_gap_results = []
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(check_drug_has_photos, name): name for name in _missing_common_names}
    for i, fut in enumerate(as_completed(futures), 1):
        _gap_results.append(fut.result())
        if i % 50 == 0:
            print(f"  {i}/{len(_missing_common_names)} checked")

_no_spls = [r for r in _gap_results if r["status"] == "no_spls_at_all"]
_spls_no_photos = [r for r in _gap_results if r["status"] == "spls_but_no_photos_sampled"]
_has_photos_missed = [r for r in _gap_results if r["status"] == "has_photos"]
_failed = [r for r in _gap_results if r["status"] == "lookup_failed"]

print(f"\nResults across {len(_gap_results)} missing common drugs:")
print(f"  {len(_no_spls)} have NO SPL on DailyMed at all (genuinely unavailable there -- "
      f"mostly dietary supplements/vitamins that aren't DailyMed-regulated the same way as drugs)")
print(f"  {len(_spls_no_photos)} have SPLs but none of the sampled ones have photos "
      f"(consistent with the bulk harvest -- real DailyMed data scarcity, not a filter bug)")
print(f"  {len(_has_photos_missed)} DO have a photo DailyMed's bulk harvest missed -- "
      f"recoverable, worth a targeted follow-up harvest:")
for r in _has_photos_missed:
    print(f"    {r['drug']}: setid {r['setid']}")
if _failed:
    print(f"  {len(_failed)} lookups failed (network/rate-limit) -- rerun this cell to retry just those.")


### V3.5.6: Investigate additional data sources (before ingesting either)

Two real candidates for getting MORE images per common pill than DailyMed's
bulk archive alone provides: NIH's C3PI/RxIMAGE archive (the same corpus
ePillID's own classes came from -- might have more than what's already
baked in) and the CURE academic pill dataset (196 classes, 20-62 images
each, real and downloadable). Neither has been verified against live data
yet, so this cell checks structure first -- same discipline as V3.B2's
smoke test before the full DailyMed parse. Run it, paste the output back,
and real ingestion code gets written against the actual structure instead
of a guess.


In [ ]:
# ============================================================================
# V3.5.6: INVESTIGATE additional data sources before committing to ingesting
#         them -- same "smoke test before full parse" discipline as V3.B2,
#         since neither of these has been verified against live data yet
#         (this sandbox can't reach NIH/Google Drive to check ahead of time).
#         Run this, then paste the output back so real ingestion code can be
#         written against the ACTUAL structure instead of a guess.
#
#   (1) NIH C3PI/RxIMAGE archive -- the same lab-quality pill photo corpus
#       ePillID's own 4,902 classes were originally built from. Might have
#       MORE images per class than what's already in ref_df, or might be
#       the exact same data already baked in -- this checks which.
#   (2) CURE pill dataset -- a real, citable academic dataset: 196 pill
#       classes, 20-62 images per class (github.com/suiyiling/
#       Few-shot-pill-recognition). Multi-image-per-class is exactly what
#       the new OTC/Rx classes are missing, IF CURE's classes can be mapped
#       to real drug identities (NDC/generic name) -- unconfirmed until we
#       see its actual folder structure and metadata.
# ============================================================================
import json as _json
import urllib.request as _ur

print("=" * 70)
print("(1) NIH C3PI/RxIMAGE archive")
print("=" * 70)
try:
    # datadiscovery.nlm.nih.gov looks like a Socrata open-data portal --
    # "5jdf-gdqh" is a Socrata dataset id, which usually exposes a JSON API
    # at /resource/<id>.json regardless of what the human-facing page shows.
    url = "https://datadiscovery.nlm.nih.gov/resource/5jdf-gdqh.json?$limit=5"
    req = _ur.Request(url, headers={"User-Agent": "pill-id-research/1.0"})
    with _ur.urlopen(req, timeout=20) as resp:
        sample = _json.load(resp)
    print(f"Reached the Socrata API -- got {len(sample)} sample row(s).")
    if sample:
        print("Fields present:", sorted(sample[0].keys()))
        print("First row:", _json.dumps(sample[0], indent=2)[:1500])
    # Get a total row count too, if the API supports it.
    try:
        count_url = "https://datadiscovery.nlm.nih.gov/resource/5jdf-gdqh.json?$select=count(*)"
        with _ur.urlopen(_ur.Request(count_url, headers={"User-Agent": "pill-id-research/1.0"}), timeout=20) as resp:
            print("Total row count:", _json.load(resp))
    except Exception as e:
        print(f"(couldn't get a total count: {e})")
except Exception as e:
    print(f"Couldn't reach the C3PI Socrata API ({e}). Try opening "
          f"https://datadiscovery.nlm.nih.gov/Chemicals-and-Drugs/Computational-Photography-Project-for-Pill-Identif/5jdf-gdqh "
          f"directly in a browser and report what you see -- specifically "
          f"whether it lists a bulk download (CSV/zip) and how many "
          f"pills/images it covers.")

print()
print("=" * 70)
print("(2) CURE pill dataset (Google Drive)")
print("=" * 70)
try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "-U", "gdown"], check=True)
    import gdown
    CURE_DIR = Path("/content/cure_dataset_probe")
    CURE_DIR.mkdir(parents=True, exist_ok=True)
    # remaining_ok=True: Google's folder-listing API caps a single directory at
    # 50 entries -- without this, gdown raises the moment it hits a bigger
    # folder (which is exactly what happened: it got through class 0-49's
    # `top`/`bottom`/`Reference` folders fine, then a `Customer` folder with
    # more than 50 real-world photos in it broke the walk). With this flag it
    # just takes the first 50 from an oversized folder and keeps going instead
    # of aborting the whole download.
    gdown.download_folder(
        id="1dcqUaTSepplc4GAUC05mr9iReWVqaThN",
        output=str(CURE_DIR),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )
    all_files = list(CURE_DIR.rglob("*"))
    print(f"\nDownloaded/found {len(all_files)} entries under {CURE_DIR}.")
    top_level = sorted({p.relative_to(CURE_DIR).parts[0] for p in all_files if p.relative_to(CURE_DIR).parts})
    print("Top-level entries:", top_level[:30])
    sample_files = [str(p.relative_to(CURE_DIR)) for p in all_files if p.is_file()][:20]
    print("Sample file paths:", sample_files)
    metadata_candidates = [p for p in all_files if p.suffix.lower() in (".csv", ".json", ".txt", ".xlsx")]
    print("Possible metadata/label files:", [str(p.relative_to(CURE_DIR)) for p in metadata_candidates][:20])
    # Real structure found on a live run: Pill_Images/<class_id>/<top|bottom>/
    # {Customer (many real-world photos), Reference (1 lab-quality photo)} --
    # per-class-id counts, to sanity check coverage once downloaded.
    class_dirs = sorted({p.parts[len(CURE_DIR.parts)+1] for p in all_files
                          if len(p.parts) > len(CURE_DIR.parts) + 1 and p.parts[len(CURE_DIR.parts)].startswith("Pill_Images")},
                         key=lambda x: (len(x), x))
    print(f"Pill class ids found: {len(class_dirs)} -- {class_dirs[:10]}{'...' if len(class_dirs) > 10 else ''}")
except Exception as e:
    print(f"Couldn't auto-download the CURE dataset folder ({e}).")
    print("Manual fallback: open "
          "https://drive.google.com/drive/folders/1dcqUaTSepplc4GAUC05mr9iReWVqaThN "
          "in a browser, look at the folder structure yourself, and report back "
          "what you see (folder names, whether filenames/metadata identify the "
          "actual drug per class, image count per class).")


In [ ]:
# ============================================================================
# V3.6: Build otc_df -- one row per kept pill image. Label = the product NDC,
#       so different manufacturers of the same generic drug are distinct
#       classes (that's the point: telling one company's ibuprofen from
#       another's). label_idx is offset by N_CLASSES so it never collides
#       with an ePillID class index.
# ============================================================================
setid_to_products = {}
for p in qualifying_products:
    setid_to_products.setdefault(p["setid"], []).append(p)

# Reference-image filenames must START WITH the NDC (matching ePillID's own
# convention) because the deployed backend parses NDC *from the filename*,
# not from a separate field (backend/app/drug_names.py::ndc_from_ref_path).
# Downloaded files are named by setid, not NDC -- copy each into an
# NDC-prefixed name here. Without this, every OTC catalog entry
# (name/imprint/color) would silently come back empty at serving time even
# though ndc_names.json has the right data, because the lookup key would be
# wrong -- caught this before it shipped.
import shutil as _shutil
NAMED_IMAGES_DIR = Path(OTC_CFG.otc_images_dir) / "named"
NAMED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

otc_rows = []
for setid, files in media_by_setid.items():
    prods = setid_to_products.get(setid)
    if not prods:
        continue
    prod = prods[0]  # one SPL can list several NDCs (pack sizes); take the first as canonical
    ndc = prod["ndc"]
    for fpath in files:
        if fpath not in kept:
            continue
        src_path = Path(fpath)
        named_path = NAMED_IMAGES_DIR / f"{ndc}_{setid}_{src_path.name}"
        if not named_path.exists():
            _shutil.copyfile(src_path, named_path)
        otc_rows.append({
            "abs_path": str(named_path),
            "label_str": ndc,                  # NDC string -- keeps drug_names.py's ndc_from_ref_path working
            "ndc": ndc,
            "brand_name": prod["brand_name"],
            "generic_name": prod["generic_name"],
            "labeler_name": prod["labeler_name"],
            "setid": setid,
            "domain": prod.get("domain", "otc"),   # "rx" | "otc" | "unknown" -- see V3.B2/V3.B3
            "is_front": True,
        })

otc_df_raw = pd.DataFrame(otc_rows).drop_duplicates("abs_path")

# Relaxed inclusion rule (was: drop every class with <2 images, which throws
# away most of a Rx/OTC bulk harvest -- most SPLs submit exactly one pill
# photo). Retrieval only needs ONE reference embedding per class to match a
# scan against -- so singleton-image classes go straight into the reference
# gallery below; only the internal top-k accuracy check needs >=2 images
# (one held out as a query, one kept as reference), so it runs on that
# eval-eligible subset only rather than gating inclusion for every class.
MIN_IMAGES_FOR_EVAL = 2
class_counts = otc_df_raw["label_str"].value_counts()
eval_eligible_labels = set(class_counts[class_counts >= MIN_IMAGES_FOR_EVAL].index)

otc_label_encoder = LabelEncoder()
otc_label_encoder.fit(otc_df_raw["label_str"])
otc_df_raw["label_idx"] = otc_label_encoder.transform(otc_df_raw["label_str"]) + N_CLASSES
N_OTC_CLASSES = len(otc_label_encoder.classes_)
N_CLASSES_MERGED = N_CLASSES + N_OTC_CLASSES
n_eval_eligible_classes = len(eval_eligible_labels)

print(f"otc_df_raw: {otc_df_raw.shape[0]} images across {N_OTC_CLASSES} product classes total -- "
      f"{n_eval_eligible_classes} have >=2 images and get an internal accuracy check below, "
      f"{N_OTC_CLASSES - n_eval_eligible_classes} have exactly 1 image and go straight into the "
      f"reference gallery without a self-eval number.")
if "domain" in otc_df_raw.columns:
    for dom, sub in otc_df_raw.groupby("domain"):
        print(f"  {dom}: {sub['label_str'].nunique()} classes, {len(sub)} images")

otc_df_raw = otc_df_raw.sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)
is_eval_eligible = otc_df_raw["label_str"].isin(eval_eligible_labels)
otc_query_df = otc_df_raw[is_eval_eligible].groupby("label_idx", group_keys=False).head(OTC_CFG.otc_query_holdout_per_class)
otc_ref_df = otc_df_raw.drop(otc_query_df.index).reset_index(drop=True)
otc_query_df = otc_query_df.reset_index(drop=True)
print(f"otc_ref_df: {otc_ref_df.shape[0]}, otc_query_df: {otc_query_df.shape[0]}")


In [ ]:
# ============================================================================
# V3.7: Extract DINOv2 448px TTA features for the OTC images and project them
#       with head_aug -- the SAME projection head the FastAPI backend loads
#       in production (backend/app/classifier.py loads a single
#       best_projection_head.pt, not the dual-head boosted ensemble above).
# ============================================================================
OTC_REF_FEAT_PATH = Path(OTC_CFG.otc_cache_root) / "otc_ref_feat_448.pt"
OTC_QUERY_FEAT_PATH = Path(OTC_CFG.otc_cache_root) / "otc_query_feat_448.pt"

otc_ref_feat_448 = extract_dinov2_features_448_tta(otc_ref_df, OTC_REF_FEAT_PATH, n_views=3)
otc_query_feat_448 = extract_dinov2_features_448_tta(otc_query_df, OTC_QUERY_FEAT_PATH, n_views=3)

with torch.no_grad():
    otc_ref_proj = apply_projection_head(head_aug, otc_ref_feat_448)
    otc_query_proj = apply_projection_head(head_aug, otc_query_feat_448)

print("otc_ref_proj emb:", otc_ref_proj["emb"].shape)
print("otc_query_proj emb:", otc_query_proj["emb"].shape)


In [ ]:
# ============================================================================
# V3.8: Merge the ePillID and OTC reference galleries and evaluate top-k
#       retrieval accuracy separately on each subset (regression check +
#       new-capability check) against the SAME merged gallery.
# ============================================================================
merged_ref_emb = torch.cat([ref_aug_proj_448["emb"], otc_ref_proj["emb"]], dim=0)
merged_ref_label_idx = torch.cat([ref_aug_proj_448["label_idx"], otc_ref_proj["label_idx"]], dim=0)
merged_ref_abs_paths = list(ref_aug_proj_448["abs_path"]) + list(otc_ref_proj["abs_path"])

print(f"Merged reference gallery: {merged_ref_emb.shape[0]} embeddings, "
      f"{N_CLASSES} ePillID classes + {N_OTC_CLASSES} OTC classes = {N_CLASSES_MERGED} total")

def eval_subset(query_proj, name):
    scores = label_score_matrix(query_proj["emb"], merged_ref_emb, merged_ref_label_idx,
                                 num_classes=N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    labels = query_proj["label_idx"]
    m = topk_accuracy_from_scores(scores, labels, topk=(1, 3, 5, 10))
    print(f"[{name}] n={labels.shape[0]}  " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))
    return m

print("=== Regression check: ePillID test queries against the MERGED gallery ===")
epillid_merged_metrics = eval_subset(
    {"emb": test_aug_proj["emb"], "label_idx": test_feat_448["label_idx"]},
    "ePillID test (merged gallery)",
)

print("\n=== New capability: OTC held-out queries against the MERGED gallery ===")
otc_merged_metrics = eval_subset(otc_query_proj, "OTC test (merged gallery)")

print("\nTarget: OTC top5 >= 0.90 and OTC top10 close to 1.0. If short, the biggest "
      "levers are: (1) raise max_spls_to_process / re-run the bulk harvest for more "
      "images per manufacturer -- a class with only 1 reference + 1 query photo is a "
      "much harder 1-shot problem than ePillID's dual-view setup; (2) raise "
      "min_pill_photo_score precision so junk images don't pollute the gallery; "
      "(3) the fine-tune option noted below.")

# This is the number that actually answers "does this fix the doctors' feedback" --
# levothyroxine/rosuvastatin/rabeprazole are Rx, so the blended OTC number above
# can look fine while Rx coverage is still zero. Only meaningful if the bulk path
# (V3.B0-V3.B3) was run with Rx archives included -- REST-path-only otc_query_df
# has no "rx" domain rows and this block just prints nothing extra.
if "domain" in otc_query_df.columns and otc_query_df["domain"].nunique() > 1:
    assert len(otc_query_proj["emb"]) == len(otc_query_df), (
        f"otc_query_proj has {len(otc_query_proj['emb'])} rows but otc_query_df has "
        f"{len(otc_query_df)} -- they've gone out of sync (e.g. V3.6 reran and rebuilt "
        f"otc_query_df but V3.7's cached features weren't recomputed to match). Rerun "
        f"V3.7 before this cell."
    )
    print("\n=== Breakdown by domain (rx vs otc) ===")
    for dom, sub in otc_query_df.groupby("domain"):
        if len(sub) == 0:
            continue
        idx = sub.index.to_numpy()
        dom_proj = {"emb": otc_query_proj["emb"][idx], "label_idx": otc_query_proj["label_idx"][idx]}
        eval_subset(dom_proj, f"{dom} test (merged gallery)")


### V3.8.5: Recalibration fine-tune

The zero-shot merge above carries the new OTC/Rx classes on an embedding
space that was only ever trained to discriminate the original 4,902 ePillID
classes -- nothing has taught it to separate the new ones yet, which is why
their accuracy lags. This continues training `head_aug`'s projection layer
(backbone stays frozen -- this is fast, no DINOv2 forward passes, just a
small MLP over already-extracted embeddings) on BOTH old and new classes
together, using ArcFace-margin softmax so even classes with a single
reference image get a real training signal (unlike triplet/contrastive
losses, which need >=2 samples of a class in the same batch to work).

Training data: ePillID's `ref_feat_448` + `val_feat_448` (old classes,
never the held-out `test_feat_448`) and the common-drug-filtered
`otc_ref_feat_448` (new classes, never the held-out `otc_query_feat_448`)
-- so the eval numbers below stay honest, not memorized.


In [ ]:
# ============================================================================
# V3.8.5 (recalibration fine-tune): continue training the projection head so
# it actually learns to discriminate the new OTC/Rx classes, not just carry
# them zero-shot. See the markdown above for what data this trains on and
# why ArcFace (not triplet/SupCon) is the right loss for a mostly-1-shot
# class distribution.
# ============================================================================
import copy

FT_CFG = {
    "epochs": 60,
    "batch_size": 256,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "arcface_margin": 0.20,
    "arcface_scale": 30.0,
    "eval_every": 5,
    # A real 250-epoch run showed WHY this matters: OTC/Rx accuracy plateaued
    # by epoch ~15-55 (never broke ~64% top5) while ePillID collapsed from
    # 86% to 66% by epoch 250 -- classic catastrophic forgetting, not
    # undertraining. distill_weight penalizes the fine-tuned head's old-class
    # embeddings for drifting away from where the ORIGINAL head_aug placed
    # them, freeing up capacity to improve new classes without destroying
    # what already worked. 0.0 reproduces the old (collapsing) behavior.
    "distill_weight": 5.0,
}

train_emb = torch.cat([ref_feat_448["emb"], val_feat_448["emb"], otc_ref_feat_448["emb"]], dim=0).float()
train_labels = torch.cat([ref_feat_448["label_idx"], val_feat_448["label_idx"], otc_ref_feat_448["label_idx"]], dim=0).long()
print(f"Fine-tune training set: {train_emb.shape[0]} embeddings across {N_CLASSES_MERGED} classes "
      f"({ref_feat_448['emb'].shape[0] + val_feat_448['emb'].shape[0]} ePillID + "
      f"{otc_ref_feat_448['emb'].shape[0]} OTC/Rx)")
is_old_class_sample = train_labels < N_CLASSES

# Cache the ORIGINAL (pre-finetune, N_CLASSES-sized) head_aug the first time
# this cell runs this session -- it reassigns head_aug to the fine-tuned
# (N_CLASSES_MERGED-sized) head at the bottom, so re-running this cell later
# (e.g. with more epochs) would otherwise warm-start from an already
# fine-tuned head instead of the true zero-shot baseline. Guarded so it only
# captures once; if head_aug is ALREADY the bigger fine-tuned head (e.g. you
# restarted this cell without first restoring head_aug from its checkpoint),
# this raises rather than silently caching the wrong thing.
if "_pretrained_head_aug_state" not in dir():
    assert head_aug.arc_weight.shape[0] == N_CLASSES, (
        f"head_aug has {head_aug.arc_weight.shape[0]} classes, expected {N_CLASSES} -- "
        f"it looks like it's already the fine-tuned head from a previous run. Restore the "
        f"original head_aug from its checkpoint (AUG_HEAD_CKPT) before running this cell."
    )
    import copy as _copy
    _pretrained_head_aug_state = _copy.deepcopy(head_aug.state_dict())
    print("Cached the original (pre-finetune) head_aug for this session -- future reruns of "
          "this cell will always warm-start from this same baseline, not from a previous "
          "fine-tune's result.")

_pretrained_head = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES).to(DEVICE)
_pretrained_head.load_state_dict(_pretrained_head_aug_state)
_pretrained_head.eval()

with torch.no_grad():
    # Frozen target embeddings for the distillation term -- only ever read,
    # never trained, so computing this once up front is exact and cheap.
    original_train_emb = F.normalize(_pretrained_head.proj(train_emb.to(DEVICE)), dim=1).cpu()

head_finetuned = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES_MERGED).to(DEVICE)
head_finetuned.proj.load_state_dict(_pretrained_head.proj.state_dict())
with torch.no_grad():
    head_finetuned.arc_weight[:N_CLASSES] = _pretrained_head.arc_weight
print(f"Warm-started proj weights from the cached original head_aug; copied {N_CLASSES} original "
      f"classes' arc_weight rows, {N_CLASSES_MERGED - N_CLASSES} new classes' rows randomly initialized.")


def arcface_logits(emb, arc_weight, labels, margin, scale):
    cos_theta = F.linear(F.normalize(emb, dim=1), F.normalize(arc_weight, dim=1)).clamp(-1 + 1e-7, 1 - 1e-7)
    theta = torch.acos(cos_theta)
    target_logit = torch.cos(theta + margin)
    one_hot = F.one_hot(labels, num_classes=arc_weight.shape[0]).float()
    return (one_hot * target_logit + (1.0 - one_hot) * cos_theta) * scale


def recompute_projections(head):
    with torch.no_grad():
        return {
            "ref_aug_proj_448": apply_projection_head(head, ref_feat_448),
            "test_aug_proj": apply_projection_head(head, test_feat_448),
            "otc_ref_proj": apply_projection_head(head, otc_ref_feat_448),
            "otc_query_proj": apply_projection_head(head, otc_query_feat_448),
        }


def quick_eval(head, label=""):
    proj = recompute_projections(head)
    merged_emb = torch.cat([proj["ref_aug_proj_448"]["emb"], proj["otc_ref_proj"]["emb"]], dim=0)
    merged_labels = torch.cat([proj["ref_aug_proj_448"]["label_idx"], proj["otc_ref_proj"]["label_idx"]], dim=0)
    epillid_scores = label_score_matrix(proj["test_aug_proj"]["emb"], merged_emb, merged_labels,
                                         N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    epillid_m = topk_accuracy_from_scores(epillid_scores, test_feat_448["label_idx"], topk=(1, 5, 10))
    otc_scores = label_score_matrix(proj["otc_query_proj"]["emb"], merged_emb, merged_labels,
                                     N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    otc_m = topk_accuracy_from_scores(otc_scores, proj["otc_query_proj"]["label_idx"], topk=(1, 5, 10))
    print(f"{label}  ePillID top5={epillid_m['top5_acc']:.4f} top10={epillid_m['top10_acc']:.4f}  |  "
          f"OTC/Rx top5={otc_m['top5_acc']:.4f} top10={otc_m['top10_acc']:.4f}")
    return epillid_m, otc_m, proj


print("\nBefore fine-tuning (the true original head_aug, not any previous fine-tune's result):")
baseline_epillid_m, baseline_otc_m, _ = quick_eval(_pretrained_head, "  [epoch 0, original head_aug]")
baseline_epillid_top5 = baseline_epillid_m["top5_acc"]

optimizer = torch.optim.AdamW(head_finetuned.parameters(), lr=FT_CFG["lr"], weight_decay=FT_CFG["weight_decay"])
n_samples = train_emb.shape[0]
best_state, best_score = None, -1e9
history = []

for epoch in range(1, FT_CFG["epochs"] + 1):
    head_finetuned.train()
    perm = torch.randperm(n_samples)
    total_loss = 0.0
    for start in range(0, n_samples, FT_CFG["batch_size"]):
        idx = perm[start:start + FT_CFG["batch_size"]]
        x = train_emb[idx].to(DEVICE)
        y = train_labels[idx].to(DEVICE)
        emb = F.normalize(head_finetuned.proj(x), dim=1)
        logits = arcface_logits(emb, head_finetuned.arc_weight, y, FT_CFG["arcface_margin"], FT_CFG["arcface_scale"])
        ce_loss = F.cross_entropy(logits, y)

        old_mask = is_old_class_sample[idx]
        if FT_CFG["distill_weight"] > 0 and old_mask.any():
            target = original_train_emb[idx][old_mask].to(DEVICE)
            distill_loss = (1.0 - (emb[old_mask] * target).sum(dim=1)).mean()
        else:
            distill_loss = torch.tensor(0.0, device=DEVICE)
        loss = ce_loss + FT_CFG["distill_weight"] * distill_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.shape[0]
    avg_loss = total_loss / n_samples

    if not math.isfinite(avg_loss):
        print(f"  [epoch {epoch}] loss diverged (NaN/Inf) -- stopping early. Try a lower FT_CFG['lr'].")
        break

    if epoch % FT_CFG["eval_every"] == 0 or epoch == FT_CFG["epochs"]:
        head_finetuned.eval()
        epillid_m, otc_m, _ = quick_eval(head_finetuned, f"  [epoch {epoch}, loss={avg_loss:.4f}]")
        # Reward new-class accuracy, but penalize regressing ePillID below its
        # own zero-shot baseline -- the point is adding capability, not
        # trading away what already worked.
        score = (otc_m["top5_acc"] + otc_m["top10_acc"]) - max(0.0, baseline_epillid_top5 - epillid_m["top5_acc"]) * 2
        history.append({"epoch": epoch, "loss": avg_loss, **{f"epillid_{k}": v for k, v in epillid_m.items()},
                         **{f"otc_{k}": v for k, v in otc_m.items()}, "score": score})
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(head_finetuned.state_dict())
            print(f"    ^ new best (score={score:.4f})")
    else:
        print(f"  [epoch {epoch}] loss={avg_loss:.4f}")

if best_state is None:
    print("\nNo epoch beat the -1e9 floor (likely diverged immediately) -- keeping head_aug as-is.")
else:
    head_finetuned.load_state_dict(best_state)
    print(f"\nLoaded best checkpoint (score={best_score:.4f}). Re-projecting everything with it...")
    head_aug = head_finetuned
    proj = recompute_projections(head_aug)
    ref_aug_proj_448 = proj["ref_aug_proj_448"]
    test_aug_proj = proj["test_aug_proj"]
    otc_ref_proj = proj["otc_ref_proj"]
    otc_query_proj = proj["otc_query_proj"]
    print("head_aug now points at the fine-tuned head -- rerun V3.8 above to see final before/after "
          "numbers side by side, then continue to V3.9.")
    if history:
        print("\nHistory (epoch, loss, ePillID top5/top10, OTC/Rx top5/top10, score):")
        for h in history:
            print(f"  {h['epoch']:>3}  loss={h['loss']:.4f}  epillid={h['epillid_top5_acc']:.4f}/{h['epillid_top10_acc']:.4f}  "
                  f"otc={h['otc_top5_acc']:.4f}/{h['otc_top10_acc']:.4f}  score={h['score']:.4f}")


### V3.8.6: LoRA backbone fine-tune (the real lever past the head-only ceiling)

V3.8.5 proved the projection-head-only approach plateaus around 60-64% top5
on OTC/Rx no matter how it's tuned -- because it can only ever move the
classification boundary on top of DINOv2 features that were never trained
to distinguish these specific pills. This cell unfreezes a small part of
the backbone itself via LoRA (rank-decomposition adapters on the attention
`query`/`value` projections -- the same technique this project's own
checkpoints show was already used once, `epillid_dinov2_fusion_finetune_
lora_...`), so the actual visual features can adapt, not just the boundary.

Unlike V3.8.5, this needs REAL images with augmentation each step (not
cached embeddings) -- LoRA changes what the backbone computes, so a fixed
pre-extracted embedding is no longer valid input. This is real GPU time:
expect tens of minutes per handful of epochs, not the near-instant epochs
V3.8.5 had.

**"Combined accuracy" is reported two ways every eval, and both are printed
every time (no picking whichever looks better):** micro-averaged (weighted
by each domain's actual query count -- 745 ePillID vs. 233 OTC/Rx, so it's
naturally dominated by ePillID) and macro-averaged (the two domains
weighted equally, which is the harder, more honest read of "does this
actually work well on the new drugs too"). Target: close to 0.80 top5 /
0.90 top10 on BOTH.


In [ ]:
# ============================================================================
# V3.8.6: LoRA backbone fine-tune. Unfreezes DINOv2's attention query/value
# projections via low-rank adapters (peft) instead of just the projection
# head -- lets the actual visual features adapt to the new classes, which is
# what V3.8.5 (head-only) structurally could not do. Uses REAL images with
# random augmentation each step, not cached embeddings.
# ============================================================================
import copy
import torchvision.transforms as _T
from transformers import AutoImageProcessor, AutoModel
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

# Self-contained (doesn't assume V3.8.5 ran first, so this cell works even
# if you skip straight to backbone fine-tuning after restoring a checkpoint).
def arcface_logits(emb, arc_weight, labels, margin, scale):
    cos_theta = F.linear(F.normalize(emb, dim=1), F.normalize(arc_weight, dim=1)).clamp(-1 + 1e-7, 1 - 1e-7)
    theta = torch.acos(cos_theta)
    target_logit = torch.cos(theta + margin)
    one_hot = F.one_hot(labels, num_classes=arc_weight.shape[0]).float()
    return (one_hot * target_logit + (1.0 - one_hot) * cos_theta) * scale


BB_FT_CFG = {
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["query", "value"],
    "epochs": 12,
    "batch_size": 8,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "arcface_margin": 0.20,
    "arcface_scale": 30.0,
    "distill_weight": 5.0,
    "eval_every": 2,
    "time_budget_seconds": 5400,  # 90 min hard ceiling -- stops gracefully, keeps the best checkpoint seen
}

# ---- 1. Build the raw-image training set (old + new classes) -------------
bb_train_df = pd.concat([
    ref_df[["abs_path", "label_idx", "is_front", "label_str"]],
    val_query_df[["abs_path", "label_idx", "is_front", "label_str"]],
    otc_ref_df[["abs_path", "label_idx", "is_front", "label_str"]],
], ignore_index=True)
bb_is_old_class = (bb_train_df["label_idx"] < N_CLASSES).to_numpy()
print(f"LoRA fine-tune training set: {len(bb_train_df)} raw images "
      f"({bb_is_old_class.sum()} ePillID + {(~bb_is_old_class).sum()} OTC/Rx)")

_bb_augment = _T.Compose([
    _T.RandomResizedCrop(448, scale=(0.7, 1.0)),
    _T.RandomHorizontalFlip(p=0.5),
    _T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
])


class BBTrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["abs_path"]).convert("RGB")
        return {"image": _bb_augment(img), "label_idx": int(row["label_idx"]), "row_idx": idx}


def _bb_collate(batch):
    return {
        "images": [b["image"] for b in batch],
        "label_idx": torch.tensor([b["label_idx"] for b in batch], dtype=torch.long),
        "row_idx": torch.tensor([b["row_idx"] for b in batch], dtype=torch.long),
    }


bb_loader = DataLoader(BBTrainDataset(bb_train_df), batch_size=BB_FT_CFG["batch_size"], shuffle=True,
                        num_workers=CFG.num_workers, collate_fn=_bb_collate, drop_last=True)

# ---- 2. Precompute FROZEN teacher embeddings for old-class images ONLY ---
# One clean (unaugmented) pass through the ORIGINAL backbone + original head
# -- this is the distillation anchor, computed once since it never changes.
_processor = AutoImageProcessor.from_pretrained(CFG.dinov2_model_id)
_frozen_backbone = AutoModel.from_pretrained(CFG.dinov2_model_id).to(DEVICE).eval()
for p in _frozen_backbone.parameters():
    p.requires_grad = False

if "_pretrained_head_aug_state" not in dir():
    assert head_aug.arc_weight.shape[0] == N_CLASSES, (
        f"head_aug has {head_aug.arc_weight.shape[0]} classes, expected {N_CLASSES} -- restore the "
        f"original head_aug from its checkpoint before running this cell."
    )
    _pretrained_head_aug_state = copy.deepcopy(head_aug.state_dict())

_pretrained_head = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES).to(DEVICE)
_pretrained_head.load_state_dict(_pretrained_head_aug_state)
_pretrained_head.eval()

print("Precomputing frozen teacher embeddings for old-class training images (one pass, no augmentation)...")
bb_teacher_emb = torch.zeros(len(bb_train_df), CFG.proj_embedding_dim)
with torch.no_grad():
    old_idx_all = np.where(bb_is_old_class)[0]
    for start in tqdm(range(0, len(old_idx_all), 32), desc="Teacher embeddings"):
        chunk_idx = old_idx_all[start:start + 32]
        imgs = [Image.open(bb_train_df.iloc[i]["abs_path"]).convert("RGB").resize((448, 448)) for i in chunk_idx]
        inputs = _processor(images=imgs, return_tensors="pt").to(DEVICE)
        feat = _frozen_backbone(**inputs).last_hidden_state[:, 0]
        proj = F.normalize(_pretrained_head.proj(feat), dim=1).cpu()
        bb_teacher_emb[chunk_idx] = proj
del _frozen_backbone
cleanup_cuda()

# ---- 3. Wrap the backbone with LoRA adapters, warm-start the head --------
_base_backbone = AutoModel.from_pretrained(CFG.dinov2_model_id).to(DEVICE)
_lora_config = LoraConfig(r=BB_FT_CFG["lora_r"], lora_alpha=BB_FT_CFG["lora_alpha"],
                           target_modules=BB_FT_CFG["target_modules"],
                           lora_dropout=BB_FT_CFG["lora_dropout"], bias="none")
backbone_lora = get_peft_model(_base_backbone, _lora_config)
backbone_lora.print_trainable_parameters()

bb_head = ProjectionHead(in_dim, CFG.proj_hidden_dim, CFG.proj_embedding_dim, N_CLASSES_MERGED).to(DEVICE)
bb_head.proj.load_state_dict(_pretrained_head.proj.state_dict())
with torch.no_grad():
    bb_head.arc_weight[:N_CLASSES] = _pretrained_head.arc_weight

bb_optimizer = torch.optim.AdamW(
    list(p for p in backbone_lora.parameters() if p.requires_grad) + list(bb_head.parameters()),
    lr=BB_FT_CFG["lr"], weight_decay=BB_FT_CFG["weight_decay"],
)


def bb_recompute_projections():
    """Re-extract 448px TTA features with the CURRENT LoRA backbone + head,
    matching the same eval path V3.8/V3.8.5 use (label_score_matrix etc)."""
    backbone_lora.eval()
    bb_head.eval()

    def _extract(df):
        embs, labels = [], []
        with torch.no_grad():
            for start in range(0, len(df), 32):
                chunk = df.iloc[start:start + 32]
                imgs = [Image.open(p).convert("RGB").resize((448, 448)) for p in chunk["abs_path"]]
                inputs = _processor(images=imgs, return_tensors="pt").to(DEVICE)
                feat = backbone_lora(**inputs).last_hidden_state[:, 0]
                proj = F.normalize(bb_head.proj(feat), dim=1).cpu()
                embs.append(proj)
                labels.append(torch.as_tensor(chunk["label_idx"].to_numpy(), dtype=torch.long))
        return {"emb": torch.cat(embs), "label_idx": torch.cat(labels)}

    return {
        "ref_aug_proj_448": _extract(ref_df),
        "test_aug_proj": _extract(test_query_df),
        "otc_ref_proj": _extract(otc_ref_df),
        "otc_query_proj": _extract(otc_query_df),
    }


def bb_quick_eval(label=""):
    proj = bb_recompute_projections()
    merged_emb = torch.cat([proj["ref_aug_proj_448"]["emb"], proj["otc_ref_proj"]["emb"]], dim=0)
    merged_labels = torch.cat([proj["ref_aug_proj_448"]["label_idx"], proj["otc_ref_proj"]["label_idx"]], dim=0)
    epillid_scores = label_score_matrix(proj["test_aug_proj"]["emb"], merged_emb, merged_labels,
                                         N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    epillid_m = topk_accuracy_from_scores(epillid_scores, proj["test_aug_proj"]["label_idx"], topk=(1, 5, 10))
    otc_scores = label_score_matrix(proj["otc_query_proj"]["emb"], merged_emb, merged_labels,
                                     N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    otc_m = topk_accuracy_from_scores(otc_scores, proj["otc_query_proj"]["label_idx"], topk=(1, 5, 10))
    n_e, n_o = proj["test_aug_proj"]["label_idx"].shape[0], proj["otc_query_proj"]["label_idx"].shape[0]
    micro = {k: (n_e * epillid_m[k] + n_o * otc_m[k]) / (n_e + n_o) for k in epillid_m}
    macro = {k: (epillid_m[k] + otc_m[k]) / 2 for k in epillid_m}
    print(f"{label}  ePillID top5={epillid_m['top5_acc']:.4f} top10={epillid_m['top10_acc']:.4f}  |  "
          f"OTC/Rx top5={otc_m['top5_acc']:.4f} top10={otc_m['top10_acc']:.4f}  |  "
          f"COMBINED micro top5={micro['top5_acc']:.4f} top10={micro['top10_acc']:.4f}  "
          f"macro top5={macro['top5_acc']:.4f} top10={macro['top10_acc']:.4f}")
    return epillid_m, otc_m, micro, macro


print("\nBefore LoRA fine-tuning:")
bb_baseline_epillid, bb_baseline_otc, bb_baseline_micro, bb_baseline_macro = bb_quick_eval("  [epoch 0]")

bb_best_state = None
bb_best_score = -1e9
bb_history = []
_bb_deadline = time.time() + BB_FT_CFG["time_budget_seconds"]
_bb_timed_out = False

for epoch in range(1, BB_FT_CFG["epochs"] + 1):
    if time.time() > _bb_deadline:
        _bb_timed_out = True
        print(f"Stopped after the {BB_FT_CFG['time_budget_seconds']/60:.0f}-minute time budget -- "
              f"keeping the best checkpoint seen so far.")
        break
    backbone_lora.train()
    bb_head.train()
    total_loss = 0.0
    n_seen = 0
    for batch in tqdm(bb_loader, desc=f"LoRA epoch {epoch}"):
        if time.time() > _bb_deadline:
            _bb_timed_out = True
            break
        inputs = _processor(images=batch["images"], return_tensors="pt").to(DEVICE)
        y = batch["label_idx"].to(DEVICE)
        feat = backbone_lora(**inputs).last_hidden_state[:, 0]
        emb = F.normalize(bb_head.proj(feat), dim=1)

        logits = arcface_logits(emb, bb_head.arc_weight, y, BB_FT_CFG["arcface_margin"], BB_FT_CFG["arcface_scale"])
        ce_loss = F.cross_entropy(logits, y)

        old_mask = bb_is_old_class[batch["row_idx"].numpy()]
        if BB_FT_CFG["distill_weight"] > 0 and old_mask.any():
            target = bb_teacher_emb[batch["row_idx"][old_mask]].to(DEVICE)
            distill_loss = (1.0 - (emb[old_mask] * target).sum(dim=1)).mean()
        else:
            distill_loss = torch.tensor(0.0, device=DEVICE)
        loss = ce_loss + BB_FT_CFG["distill_weight"] * distill_loss

        bb_optimizer.zero_grad()
        loss.backward()
        bb_optimizer.step()
        total_loss += loss.item() * y.shape[0]
        n_seen += y.shape[0]

    if _bb_timed_out:
        break
    avg_loss = total_loss / max(n_seen, 1)
    if not math.isfinite(avg_loss):
        print(f"  [epoch {epoch}] loss diverged (NaN/Inf) -- stopping early. Try a lower BB_FT_CFG['lr'].")
        break

    if epoch % BB_FT_CFG["eval_every"] == 0 or epoch == BB_FT_CFG["epochs"]:
        epillid_m, otc_m, micro, macro = bb_quick_eval(f"  [epoch {epoch}, loss={avg_loss:.4f}]")
        score = macro["top5_acc"] + macro["top10_acc"]  # optimize the harder (macro) combined metric
        bb_history.append({"epoch": epoch, "loss": avg_loss, "epillid_top5": epillid_m["top5_acc"],
                            "otc_top5": otc_m["top5_acc"], "micro_top5": micro["top5_acc"],
                            "micro_top10": micro["top10_acc"], "macro_top5": macro["top5_acc"],
                            "macro_top10": macro["top10_acc"], "score": score})
        if score > bb_best_score:
            bb_best_score = score
            bb_best_state = {
                "backbone_lora": copy.deepcopy(get_peft_model_state_dict(backbone_lora)),
                "head": copy.deepcopy(bb_head.state_dict()),
            }
            print(f"    ^ new best (macro score={score:.4f})")
    else:
        print(f"  [epoch {epoch}] loss={avg_loss:.4f}")

if bb_best_state is not None:
    set_peft_model_state_dict(backbone_lora, bb_best_state["backbone_lora"])
    bb_head.load_state_dict(bb_best_state["head"])
    print(f"\nLoaded best LoRA checkpoint (macro score={bb_best_score:.4f}).")
    print("Final numbers, both combined-metric definitions:")
    bb_quick_eval("  [best checkpoint]")
    print("\nbackbone_lora and bb_head now hold the fine-tuned model. To use them for export in V3.9, "
          "the export cell needs to call backbone_lora(**inputs).last_hidden_state[:, 0] instead of "
          "the cached ref_feat_448/otc_ref_feat_448 embeddings -- see the note in V3.9 if it hasn't "
          "been updated for that yet.")
else:
    print("\nNo epoch completed a full eval before the time budget ran out -- nothing to load. "
          "Increase BB_FT_CFG['time_budget_seconds'] or reduce epochs/batch overhead and rerun.")


In [ ]:
# ============================================================================
# V3.9: Filter the merged gallery to classes with real multi-image evidence,
#       then export deployment artifacts for the FastAPI backend. Matches
#       backend/app/classifier.py's expected format exactly:
#         best_projection_head.pt   : {in_dim, num_classes, head_state_dict, label_classes}
#         deployed_ref_embeddings.pt: {embeddings, label_indices, abs_paths}
#
#       A real run found that keeping ALL new classes (including the
#       majority that only have 1 reference image) measurably hurt BOTH
#       ePillID's own accuracy (noise/distractor pollution in the shared
#       embedding space) AND the new classes' own accuracy (many are
#       inherently hard 1-shot problems) -- dropping the singleton-image
#       classes and shipping only the ones with >=2 images (`eval_eligible_
#       labels`, from V3.6) recovered several points on both, confirmed
#       against held-out queries. That's what this cell exports: fewer new
#       classes, but ones that actually work, rather than maximum raw count.
# ============================================================================
MERGED_OUT_DIR = RUN_DIR_MAIN / "merged_multi_db_export"
MERGED_OUT_DIR.mkdir(parents=True, exist_ok=True)

eval_eligible_mask_in_ref = otc_ref_df["label_str"].isin(eval_eligible_labels).to_numpy()
otc_ref_emb_filtered = otc_ref_proj["emb"][eval_eligible_mask_in_ref]
otc_ref_label_filtered = otc_ref_proj["label_idx"][eval_eligible_mask_in_ref]
otc_ref_abs_paths_filtered = [
    p for p, keep in zip(otc_ref_proj["abs_path"], eval_eligible_mask_in_ref) if keep
]

# Drop singleton classes from the exported label space entirely (not just
# from the reference pool) and remap indices so there are no gaps -- the
# deployed classifier expects a compact 0..N-1 label_classes list.
otc_kept_class_indices_offset = sorted(set(otc_ref_label_filtered.tolist()))
otc_kept_class_indices = [i - N_CLASSES for i in otc_kept_class_indices_offset]
otc_label_classes = [otc_label_encoder.classes_[i] for i in otc_kept_class_indices]

epillid_label_classes = list(label_encoder.classes_)      # length N_CLASSES, unchanged
merged_label_classes = epillid_label_classes + otc_label_classes
N_CLASSES_MERGED = len(merged_label_classes)

remap = {old: N_CLASSES + new for new, old in enumerate(otc_kept_class_indices_offset)}
otc_ref_label_remapped = torch.tensor([remap[int(x)] for x in otc_ref_label_filtered])

merged_ref_emb = torch.cat([ref_aug_proj_448["emb"], otc_ref_emb_filtered], dim=0)
merged_ref_label_idx = torch.cat([ref_aug_proj_448["label_idx"], otc_ref_label_remapped], dim=0)
merged_ref_abs_paths = list(ref_aug_proj_448["abs_path"]) + otc_ref_abs_paths_filtered

assert len(merged_label_classes) == N_CLASSES_MERGED
assert merged_ref_label_idx.max().item() < N_CLASSES_MERGED

# Sanity check right before shipping: remap otc_query_proj's labels the same
# way (queries only ever come from eval-eligible classes -- see V3.6 -- so
# every query label is guaranteed to be in `remap`) and confirm accuracy on
# EXACTLY what's about to be exported, not a stale number from an earlier run.
otc_query_label_remapped = torch.tensor([remap[int(x)] for x in otc_query_proj["label_idx"]])

def eval_subset(query_proj_emb, query_labels, name):
    scores = label_score_matrix(query_proj_emb, merged_ref_emb, merged_ref_label_idx,
                                 num_classes=N_CLASSES_MERGED, chunk_size=CFG.score_chunk_size)
    m = topk_accuracy_from_scores(scores, query_labels, topk=(1, 3, 5, 10))
    print(f"[{name}] n={query_labels.shape[0]}  " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))
    return m

print(f"Exporting {N_CLASSES_MERGED} total classes ({N_CLASSES} ePillID + {len(otc_label_classes)} new "
      f"-- dropped {len(otc_label_encoder.classes_) - len(otc_label_classes)} singleton-image classes)")
print("\n=== Accuracy on exactly what's being shipped ===")
epillid_m = eval_subset(test_aug_proj["emb"], test_feat_448["label_idx"], "ePillID test")
otc_m = eval_subset(otc_query_proj["emb"], otc_query_label_remapped, "New-class test")
n_e, n_o = test_feat_448["label_idx"].shape[0], otc_query_label_remapped.shape[0]
for k in ("top1_acc", "top3_acc", "top5_acc", "top10_acc"):
    combined = (n_e * epillid_m[k] + n_o * otc_m[k]) / (n_e + n_o)
    print(f"  combined {k} = {combined:.4f}")

# CRITICAL: export the CURRENT head_aug (whatever V3.8.5/V3.8.6 fine-tuning
# produced -- or the original zero-shot head, if you're deliberately
# skipping fine-tuning), NOT aug_ckpt. aug_ckpt is the ORIGINAL checkpoint
# loaded from disk back in the cell that first defined head_aug, frozen at
# that point forever -- exporting it here would silently ship the
# pre-finetune head while this cell's own printed eval numbers reflect
# whatever head_aug currently is. That's exactly the "what's evaluated !=
# what's deployed" gap that makes a reported accuracy number meaningless.
print(f"Exporting head_aug as-is: {head_aug.arc_weight.shape[0]} classes "
      f"({'fine-tuned (V3.8.5/V3.8.6 ran)' if head_aug.arc_weight.shape[0] == N_CLASSES_MERGED else 'zero-shot, no fine-tuning applied'}).")
deployed_head_ckpt = {
    "in_dim": in_dim,
    # Read dynamically from head_aug itself, not hardcoded to N_CLASSES --
    # a zero-shot merge (no fine-tuning) keeps head_aug at N_CLASSES (only
    # .proj is used at inference, classifier/arc_weight are never touched),
    # but a fine-tuned head (V3.8.5/V3.8.6) genuinely has N_CLASSES_MERGED
    # arc_weight/classifier rows -- reconstructing the head with the wrong
    # num_classes at serving time would fail load_state_dict on a shape
    # mismatch. This is correct either way without needing to track which
    # path was taken.
    "num_classes": head_aug.arc_weight.shape[0],
    "head_state_dict": head_aug.state_dict(),
    "label_classes": merged_label_classes,
}
torch.save(deployed_head_ckpt, MERGED_OUT_DIR / "best_projection_head.pt")

if "backbone_lora" in dir():
    print("\n*** backbone_lora is in your session (from V3.8.6) -- this export only saves the "
          "projection head. The LoRA adapter weights on the BACKBONE also need to be exported and "
          "the FastAPI backend updated to apply them at inference, or the deployed app will use the "
          "plain (non-LoRA-adapted) backbone and NOT match the accuracy numbers evaluated above. "
          "See the LoRA export cell below -- run it before deploying if you fine-tuned the backbone. ***")

deployed_ref_embeddings = {
    "embeddings": merged_ref_emb.cpu(),
    "label_indices": merged_ref_label_idx.cpu(),
    "abs_paths": merged_ref_abs_paths,
}
torch.save(deployed_ref_embeddings, MERGED_OUT_DIR / "deployed_ref_embeddings.pt")

# Zip the OTC/Rx image files themselves (mirrors ePillID_data.zip) so the
# backend can serve their thumbnails too.
otc_zip_path = MERGED_OUT_DIR / "otc_reference_images.zip"
with zipfile.ZipFile(otc_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in {str(p) for p in merged_ref_abs_paths if str(p).startswith(OTC_CFG.otc_images_dir)}:
        zf.write(p, arcname=f"otc_data/{Path(p).name}")

print("\nSaved:", [str(p) for p in MERGED_OUT_DIR.glob("*")])
print("\nTo deploy: copy best_projection_head.pt + deployed_ref_embeddings.pt into "
      "pill-id/dinov2_projection_head/, and otc_reference_images.zip into the "
      "pill-id repo root (next to ePillID_data.zip). Use V3.10 below for ndc_names.json "
      "-- NOT the raw brand_name/generic_name fields on qualifying_products, which are "
      "unreliable (see V3.B3's note).")


### V3.9.5: Export LoRA backbone adapter weights (only needed if V3.8.6 ran)

If you fine-tuned the backbone with V3.8.6, the projection-head export in
V3.9 above is not enough on its own -- the actual visual features being
served depend on the LoRA adapters too. This saves just the small adapter
weights (not the whole backbone). **The FastAPI backend also needs a
corresponding code change to load and apply these at inference** -- that
hasn't been made yet as of this notebook. Don't deploy a LoRA-fine-tuned
model until that's done, or the served app will silently use the plain,
non-adapted backbone and won't match any of the accuracy numbers evaluated
here.


In [ ]:
# ============================================================================
# V3.9.5: Export LoRA backbone adapter weights -- only relevant if V3.8.6
# was run. Skips itself if backbone_lora isn't in the session.
# ============================================================================
if "backbone_lora" not in dir():
    print("backbone_lora not in this session (V3.8.6 wasn't run, or you're shipping the zero-shot/"
          "head-only-fine-tuned model) -- nothing to export here.")
else:
    from peft import get_peft_model_state_dict

    lora_adapter_path = MERGED_OUT_DIR / "backbone_lora_adapter.pt"
    torch.save({
        "adapter_state_dict": get_peft_model_state_dict(backbone_lora),
        "lora_config": {
            "r": BB_FT_CFG["lora_r"],
            "lora_alpha": BB_FT_CFG["lora_alpha"],
            "lora_dropout": BB_FT_CFG["lora_dropout"],
            "target_modules": BB_FT_CFG["target_modules"],
        },
        "base_model_id": CFG.dinov2_model_id,
    }, lora_adapter_path)
    print(f"Saved LoRA adapter weights -> {lora_adapter_path}")
    print("\n*** Before deploying: the backend needs a matching change to load this adapter onto its "
          "DINOv2 backbone (peft.LoraConfig + peft.get_peft_model + peft.set_peft_model_state_dict, "
          "same pattern as V3.8.6) and apply it at inference. Until that's in backend/app/classifier.py, "
          "do NOT copy this file into a deployment -- it will be silently ignored and the served app "
          "will use the plain, non-adapted backbone. ***")


In [ ]:
# ============================================================================
# V3.10: Resolve real drug names for the new classes via NIH's RxNav API,
#        keyed by NDC -- NOT from the SPL XML's brand/generic/labeler
#        fields (see V3.B3's note: that extraction was unreliable and has
#        been removed). This is the exact same proven method
#        backend/scripts/build_ndc_names.py already uses for the original
#        ePillID/OTC name table, duplicated inline here (rather than
#        depending on that script being present in this Colab session) so
#        this cell is self-contained.
# ============================================================================
import urllib.error
from concurrent.futures import ThreadPoolExecutor as _TPE, as_completed as _as_completed

RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"

class _RateLimited(Exception):
    pass

def _rxnav_get(url: str, timeout: int = 20, retries: int = 5) -> dict:
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "pill-id-research/1.0"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.load(resp)
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise _RateLimited(f"{url}: {last_err}")

def _rxnav_candidates(token: str) -> list:
    parts = token.split("-")
    cands = []
    if len(parts) >= 2:
        cands.append(f"{parts[0].zfill(5)}-{parts[1].zfill(4)}")  # ndc9
    cands.append(token)
    seen = set()
    return [c for c in cands if not (c in seen or seen.add(c))]

def _rxnav_name(rxcui: str, cache: dict) -> "str | None":
    if rxcui in cache:
        return cache[rxcui]
    data = _rxnav_get(f"{RXNAV_BASE}/rxcui/{rxcui}/property.json?propName=RxNorm%20Name")
    pc = data.get("propConceptGroup", {}).get("propConcept", [])
    name = pc[0]["propValue"] if pc else None
    cache[rxcui] = name
    return name

def _rxnav_find_prop(props: dict, *substrings: str) -> "str | None":
    for prop_name, value in props.items():
        if any(s in prop_name.upper() for s in substrings) and value:
            return value
    return None

def resolve_ndc_via_rxnav(token: str, rx_cache: dict) -> "dict | None":
    for cand in _rxnav_candidates(token):
        data = _rxnav_get(f"{RXNAV_BASE}/ndcproperties.json?id={cand}&ndcstatus=ALL")
        pl = data.get("ndcPropertyList", {}).get("ndcProperty", [])
        if not pl:
            continue
        p = pl[0]
        props = {x["propName"]: x["propValue"] for x in p.get("propertyConceptList", {}).get("propertyConcept", [])}
        rxcui = p.get("rxcui") or None
        name = _rxnav_name(rxcui, rx_cache) if rxcui else None
        if not name:
            continue
        return {
            "name": name, "rxcui": rxcui,
            "imprint": props.get("IMPRINT_CODE") or None,
            "color": props.get("COLORTEXT") or None,
            "status": props.get("NDC_STATUS") or None,
            "shape": _rxnav_find_prop(props, "SHAPE"),
            "score": _rxnav_find_prop(props, "SCORE"),
        }
    return None

NDC_NAMES_IN = Path("/content/ndc_names.json")   # upload backend/app/data/ndc_names.json here first
if "ndc_lookup" in dir():
    print(f"Reusing {len(ndc_lookup)} NDC names already resolved earlier this session (V3.5.5) -- "
          f"nothing new to fetch unless the common-drug filter let through NDCs it didn't resolve.")
else:
    ndc_lookup = json.load(open(NDC_NAMES_IN)) if NDC_NAMES_IN.exists() else {}
    print(f"Loaded {len(ndc_lookup)} existing NDC names" if ndc_lookup else "No existing ndc_names.json found -- starting fresh")

tokens_to_resolve = sorted({ndc for ndc in merged_label_classes[N_CLASSES:] if ndc not in ndc_lookup})
print(f"{len(tokens_to_resolve)} new NDCs to resolve via RxNav")

_rx_cache = {}
_resolved = _rate_limited = 0
with _TPE(max_workers=5) as pool:
    futures = {pool.submit(resolve_ndc_via_rxnav, t, _rx_cache): t for t in tokens_to_resolve}
    for i, fut in enumerate(_as_completed(futures), 1):
        token = futures[fut]
        try:
            info = fut.result()
            if info:
                ndc_lookup[token] = info
                _resolved += 1
        except _RateLimited:
            _rate_limited += 1
        if i % 100 == 0:
            print(f"  {i}/{len(tokens_to_resolve)} processed, {_resolved} resolved, {_rate_limited} rate-limited")

MERGED_NDC_NAMES_OUT = MERGED_OUT_DIR / "ndc_names.json"
json.dump(ndc_lookup, open(MERGED_NDC_NAMES_OUT, "w"), indent=0, sort_keys=True)
print(f"\nResolved {_resolved}/{len(tokens_to_resolve)} new NDCs; {len(ndc_lookup)} total NDC names -> {MERGED_NDC_NAMES_OUT}")
unresolved = [t for t in merged_label_classes[N_CLASSES:] if t not in ndc_lookup]
if unresolved:
    print(f"{len(unresolved)} exported classes have no resolved name yet (not in RxNorm, or rate-limited -- "
          f"rerun this cell to retry; already-resolved NDCs are skipped automatically).")
if _rate_limited:
    print(f"WARNING: {_rate_limited} requests rate-limited this run; rerun to retry them.")
print("\nCopy this over backend/app/data/ndc_names.json.")


In [ ]:
# ============================================================================
# V3.11: Optional -- merged export for the Phase-2 client-side (ONNX/browser)
#        app (med-recognition-app).
#        WARNING: reference_embeddings.json embeds every reference vector as
#        raw JSON floats and is fetched by the browser on load. The
#        ePillID-only version is already tens of MB; adding thousands of OTC
#        vectors will make this significantly bigger. Consider capping
#        refs/class or switching to a binary float16 format before shipping
#        this to med-recognition-app -- that's a follow-up, not done here.
# ============================================================================
merged_reference_payload = {
    "ref_aug_embeddings": merged_ref_emb.cpu().numpy().astype(np.float32).tolist(),
    "ref_label_idx": merged_ref_label_idx.cpu().numpy().tolist(),
    "embedding_dim": int(merged_ref_emb.shape[1]),
    "num_reference_images": int(merged_ref_emb.shape[0]),
}
with open(MERGED_OUT_DIR / "reference_embeddings_merged.json", "w") as f:
    json.dump(merged_reference_payload, f)

merged_label_map = {}
for idx, label_str in enumerate(merged_label_classes):
    source = "epillid" if idx < N_CLASSES else "dailymed_otc"
    info = ndc_lookup.get(label_str, {})
    merged_label_map[idx] = {
        "label_str": label_str,
        "drug_name": info.get("name", label_str),
        "ndc": label_str,
        "source": source,
    }
with open(MERGED_OUT_DIR / "labels_merged.json", "w") as f:
    json.dump(merged_label_map, f, indent=2)

size_mb = (MERGED_OUT_DIR / "reference_embeddings_merged.json").stat().st_size / 1e6
print(f"reference_embeddings_merged.json: {size_mb:.1f} MB -- {merged_ref_emb.shape[0]} vectors")


# **V2 APP DEPLOYMENT**

In [ ]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.3 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# ENHANCED EXPORT - includes drug names from ndc_names.json
# Run in Colab AFTER cells 0-12 (same as export_to_onnx.py but with drug names)
# ============================================================================

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from pathlib import Path

WEB_EXPORT_DIR = OUT_DIR / "web_export_final_v2"
WEB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 448

# ============================================================================
# Load your ndc_names.json from wherever you uploaded it
# (You'll need to upload ndc_names.json to Colab first, or paste the JSON here)
# ============================================================================
# Option 1: If you upload ndc_names.json to Colab /content/ directory:
ndc_names_path = Path("/content/ndc_names.json")  # Upload this file to Colab first
if ndc_names_path.exists():
    with open(ndc_names_path) as f:
        ndc_lookup = json.load(f)
    print(f"Loaded {len(ndc_lookup)} NDC entries")
else:
    ndc_lookup = {}
    print("WARNING: ndc_names.json not found. Drug names will be missing.")

# ============================================================================
# Export models (same as before)
# ============================================================================
class BackboneForExport(nn.Module):
    def __init__(self, hf_model_id: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(hf_model_id)
        self.backbone.eval()

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        return out.last_hidden_state[:, 0, :]

print("Loading DINOv2 backbone for export...")
backbone_export = BackboneForExport(CFG.dinov2_model_id).to(DEVICE)
backbone_export.eval()

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

torch.onnx.export(
    backbone_export,
    dummy_input,
    str(WEB_EXPORT_DIR / "backbone.onnx"),
    input_names=["pixel_values"],
    output_names=["embedding"],
    dynamic_axes={"pixel_values": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=18,
    do_constant_folding=True,
)
print("Saved backbone.onnx")
del backbone_export
torch.cuda.empty_cache()

class ProjectionOnlyExport(nn.Module):
    def __init__(self, head):
        super().__init__()
        self.proj = head.proj

    def forward(self, x):
        return self.proj(x)

def export_head(head, name):
    head.eval()
    wrapper = ProjectionOnlyExport(head).to(DEVICE).eval()
    dummy = torch.randn(1, in_dim, device=DEVICE)
    torch.onnx.export(
        wrapper,
        dummy,
        str(WEB_EXPORT_DIR / f"{name}.onnx"),
        input_names=["backbone_embedding"],
        output_names=["projected_embedding"],
        dynamic_axes={"backbone_embedding": {0: "batch"}, "projected_embedding": {0: "batch"}},
        opset_version=18,
        do_constant_folding=False, # Changed to False
    )
    print(f"Saved {name}.onnx")

export_head(head_aug, "head_aug")
export_head(head_lora, "head_lora")

# Reference embeddings
with torch.no_grad():
    ref_aug_emb = F.normalize(head_aug.proj(ref_feat_448["emb"].to(DEVICE)), dim=1).cpu().numpy()
    ref_lora_emb = F.normalize(head_lora.proj(ref_feat_448["emb"].to(DEVICE)), dim=1).cpu().numpy()

ref_labels = ref_feat_448["label_idx"].cpu().numpy().tolist()

reference_payload = {
    "ref_aug_embeddings": ref_aug_emb.astype(np.float32).tolist(),
    "ref_lora_embeddings": ref_lora_emb.astype(np.float32).tolist(),
    "ref_label_idx": ref_labels,
    "embedding_dim": int(ref_aug_emb.shape[1]),
    "num_reference_images": int(ref_aug_emb.shape[0]),
}
with open(WEB_EXPORT_DIR / "reference_embeddings.json", "w") as f:
    json.dump(reference_payload, f)
print("Saved reference_embeddings.json")

# ============================================================================
# ENHANCED: labels.json with drug names from ndc_names.json
# ============================================================================
label_map = {}
for _, row in ref_df.drop_duplicates("label_idx").iterrows():
    label_idx = int(row["label_idx"])
    label_str = str(row["label_str"])

    # Try to find drug name from NDC lookup
    drug_name = label_str  # default to label_str
    ndc_code = None

    # If label_str looks like an NDC (e.g., "00123-4567-89"), look it up
    if "-" in label_str and len(label_str) > 5:
        if label_str in ndc_lookup:
            ndc_data = ndc_lookup[label_str]
            drug_name = ndc_data.get("name", label_str)
            ndc_code = label_str

    label_map[label_idx] = {
        "label_str": label_str,
        "drug_name": drug_name,
        "ndc": ndc_code,
    }

with open(WEB_EXPORT_DIR / "labels.json", "w") as f:
    json.dump(label_map, f, indent=2)
print(f"Saved labels.json with {len(label_map)} classes (drug names included)")

# Inference config
inference_config = {
    "img_size": IMG_SIZE,
    "n_tta_views": 3,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "num_classes": int(N_CLASSES),
}
with open(WEB_EXPORT_DIR / "inference_config.json", "w") as f:
    json.dump(inference_config, f, indent=2)
print("Saved inference_config.json")

# Zip
import shutil
shutil.make_archive(str(WEB_EXPORT_DIR), "zip", WEB_EXPORT_DIR)
print(f"\nDONE. Download: {str(WEB_EXPORT_DIR)}.zip")
print("\nFile sizes:")
for p in WEB_EXPORT_DIR.glob("*"):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ============================================================================
# RUN THIS IN COLAB AFTER export_to_onnx.py
# Quantizes your models to int8 (shrinks ~3x, speeds up ~2x)
# ============================================================================

!pip install onnxruntime

import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from pathlib import Path

WEB_EXPORT_DIR = Path(OUT_DIR) / "web_export_final_v2"  # must match the export cell above
QUANTIZED_DIR = Path(OUT_DIR) / "web_export_quantized_v2"
QUANTIZED_DIR.mkdir(exist_ok=True)

print("Quantizing ONNX models to int8...")

for model_name in ["backbone.onnx", "head_aug.onnx", "head_lora.onnx"]:
    input_path = str(WEB_EXPORT_DIR / model_name)
    output_path = str(QUANTIZED_DIR / model_name)

    print(f"\nQuantizing {model_name}...")
    quantize_dynamic(input_path, output_path, weight_type=QuantType.QUInt8)

    input_size = (WEB_EXPORT_DIR / model_name).stat().st_size / 1e6
    output_size = (QUANTIZED_DIR / model_name).stat().st_size / 1e6
    print(f"  {model_name}: {input_size:.1f}MB → {output_size:.1f}MB")

# Copy JSON files (they don't need quantizing)
import shutil
for json_file in ["reference_embeddings.json", "labels.json", "inference_config.json"]:
    src = WEB_EXPORT_DIR / json_file
    dst = QUANTIZED_DIR / json_file
    if src.exists():
        shutil.copy(src, dst)
        print(f"Copied {json_file}")

# Zip it up
shutil.make_archive(str(OUT_DIR / "web_export_final"), "zip", QUANTIZED_DIR)
print(f"\nDONE! Download: {str(OUT_DIR / 'web_export_final')}.zip")
print("\nFile sizes (quantized):")
for p in QUANTIZED_DIR.glob("*"):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

In [ ]:
print("=" * 60)
print("CHECK YOUR MODEL DIMENSIONS")
print("=" * 60)

# From backbone output
print(f"\nBackbone output shape:")
dummy_input = torch.randn(1, 3, 448, 448, device=DEVICE)
with torch.no_grad():
    backbone_out = AutoModel.from_pretrained(CFG.dinov2_model_id).to(DEVICE)(pixel_values=dummy_input)
backbone_emb = backbone_out.last_hidden_state[:, 0, :]
print(f"  DINOv2 CLS token shape: {backbone_emb.shape}")
backbone_dim = backbone_emb.shape[1]
print(f"  Embedding dimension: {backbone_dim}")

# Check head_aug and head_lora input expectations
print(f"\nProjection heads input expectations:")
print(f"  head_aug.proj expects input shape: (batch, {in_dim})")
print(f"  head_lora.proj expects input shape: (batch, {in_dim})")
print(f"  Actual backbone output: (batch, {backbone_dim})")

if backbone_dim != in_dim:
    print(f"\n⚠️  MISMATCH! {backbone_dim} ≠ {in_dim}")
    print(f"   The fix: in your export script, change 'in_dim' to {backbone_dim}")
else:
    print(f"\n✓ Dimensions match!")

CHECK YOUR MODEL DIMENSIONS

Backbone output shape:


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

  DINOv2 CLS token shape: torch.Size([1, 1024])
  Embedding dimension: 1024

Projection heads input expectations:
  head_aug.proj expects input shape: (batch, 1024)
  head_lora.proj expects input shape: (batch, 1024)
  Actual backbone output: (batch, 1024)

✓ Dimensions match!
